In [6]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("hf_token")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [7]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/models/strugglingug/model6/pytorch/default/1/legal_bpe_v3.pkl
/kaggle/input/models/strugglingug/model6/pytorch/default/1/val_v3.bin
/kaggle/input/models/strugglingug/model6/pytorch/default/1/train_v3.bin


In [8]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [10]:
import re
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

from datasets import load_dataset


class LegalBPETokenizer:

    def __init__(self, model_path):

        with open(model_path, "rb") as f:
            model = pickle.load(f)

        self.vocab_size = model["vocab_size"]
        self.merges = model["merges"]                     # [((a, b), new_id), ...]
        self.lowercase = model.get("lowercase", False)
        self.special_tokens = model.get("special_tokens", [])
        self.pattern = re.compile(model["pattern"])

        # pair -> rank (lower rank = learned earlier = merged first)
        self.merge_rank = {
            tuple(pair): rank
            for rank, (pair, _) in enumerate(self.merges)
        }

        # id -> bytes table (O(1) decode)
        self.token_bytes = {i: bytes([i]) for i in range(256)}
        for (l, r), new_id in self.merges:
            self.token_bytes[new_id] = self.token_bytes[l] + self.token_bytes[r]

        base = 256 + len(self.merges)
        self.special_to_id = {t: base + i for i, t in enumerate(self.special_tokens)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.eos_id = self.special_to_id.get("<|endoftext|>")

        self._special_re = (
            re.compile("(" + "|".join(re.escape(t) for t in self.special_tokens) + ")")
            if self.special_tokens else None
        )
        self._cache = {}

    def _encode_piece(self, piece_bytes):

        cached = self._cache.get(piece_bytes)
        if cached is not None:
            return cached

        ids = list(piece_bytes)

        while len(ids) > 1:
            best, best_rank = None, None
            for p in zip(ids, ids[1:]):
                r = self.merge_rank.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = p, r
            if best is None:
                break

            new_id = 256 + best_rank
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == best[0] and ids[i + 1] == best[1]:
                    out.append(new_id)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out

        if len(self._cache) < 500_000:
            self._cache[piece_bytes] = ids
        return ids

    def _encode_ordinary(self, text):

        if self.lowercase:
            text = text.lower()

        out = []
        for piece in self.pattern.findall(text):
            out.extend(self._encode_piece(piece.encode("utf-8")))
        return out

    def encode(self, text, add_eos=False):

        if self._special_re is not None:
            ids = []
            for part in self._special_re.split(text):
                if part in self.special_to_id:
                    ids.append(self.special_to_id[part])
                elif part:
                    ids.extend(self._encode_ordinary(part))
        else:
            ids = self._encode_ordinary(text)

        if add_eos:
            ids.append(self.eos_id)

        return ids

    def decode(self, ids):

        chunks = []
        for token_id in ids:
            if token_id in self.id_to_special:
                chunks.append(self.id_to_special[token_id].encode("utf-8"))
            else:
                chunks.append(self.token_bytes[token_id])

        return b"".join(chunks).decode("utf-8", errors="replace")


TOKENIZER_PATH = "/kaggle/input/models/strugglingug/model6/pytorch/default/1/legal_bpe_v3.pkl"   # <-- change this

tokenizer = LegalBPETokenizer(TOKENIZER_PATH)

print("Vocabulary size:", tokenizer.vocab_size)
print([tokenizer.decode([i]) for i in tokenizer.encode("dog ate cat")])

Vocabulary size: 30000
['d', 'og', ' ate', ' cat']


In [5]:
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

acts_dataset = load_dataset(
    "json",
    data_files={
        "train": f"hf://datasets/{HF_DATASET_REPO}/Phase_1_Pretraining/Train/train_acts.json"
    },
    split="train",
    streaming=True
)

print("Acts dataset loaded.")

Acts dataset loaded.


In [ ]:
acts_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=files["acts"],
    repo_type="dataset"
)

print(acts_path)

In [13]:
%%writefile attention.py

"""Multi-head causal self-attention, implemented from raw tensor ops."""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        max_seq_len: int,
        dropout: float = 0.0,
        attention_scale=None
    ):

        super().__init__()

        assert d_model % n_heads == 0

        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.attention_scale = (
            1.0 / math.sqrt(self.d_head)
            if attention_scale is None
            else float(attention_scale)
        )

        self.q_proj = nn.Linear(
            d_model, d_model, bias=False
        )

        self.k_proj = nn.Linear(
            d_model, d_model, bias=False
        )

        self.v_proj = nn.Linear(
            d_model, d_model, bias=False
        )

        self.out_proj = nn.Linear(
            d_model, d_model, bias=False
        )

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(
            torch.ones(
                max_seq_len,
                max_seq_len,
                dtype=torch.bool
            )
        )

        self.register_buffer(
            "causal_mask",
            causal_mask,
            persistent=False
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        B, T, C = x.shape

        # Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # (B, T, C)
        #     ↓
        # (B, T, n_heads, d_head)
        #     ↓
        # (B, n_heads, T, d_head)

        q = q.view(
            B, T, self.n_heads, self.d_head
        ).transpose(1, 2)

        k = k.view(
            B, T, self.n_heads, self.d_head
        ).transpose(1, 2)

        v = v.view(
            B, T, self.n_heads, self.d_head
        ).transpose(1, 2)

        # Attention scores
        scores = torch.matmul(
            q,
            k.transpose(-2, -1)
        )

        scores = scores * self.attention_scale

        # Causal mask
        mask = self.causal_mask[:T, :T]

        scores = scores.masked_fill(
            ~mask,
            float("-inf")
        )

        # Softmax
        attn = F.softmax(
            scores,
            dim=-1
        )

        attn = self.attn_dropout(attn)

        # Weighted sum
        out = torch.matmul(
            attn,
            v
        )

        # (B, n_heads, T, d_head)
        #       ↓
        # (B, T, C)

        out = out.transpose(
            1, 2
        ).contiguous().view(B, T, C)

        out = self.out_proj(out)

        out = self.resid_dropout(out)

        return out

Overwriting attention.py


In [14]:
%%writefile model_arch.py

"""Decoder-only Transformer LM, built from scratch."""

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

from attention import CausalSelfAttention


@dataclass
class LMConfig:

    vocab_size: int

    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4

    d_ff: int = 1024

    max_seq_len: int = 256

    dropout: float = 0.1

    pos_emb_type: str = "sinusoidal"

    attention_scale: float = None

    norm_first: bool = False


def sinusoidal_positional_encoding(
    max_len: int,
    d_model: int
) -> torch.Tensor:

    pe = torch.zeros(
        max_len,
        d_model
    )

    position = torch.arange(
        max_len,
        dtype=torch.float32
    ).unsqueeze(1)

    div_term = torch.exp(
        torch.arange(
            0,
            d_model,
            2,
            dtype=torch.float32
        )
        * (-math.log(10000.0) / d_model)
    )

    pe[:, 0::2] = torch.sin(
        position * div_term
    )

    pe[:, 1::2] = torch.cos(
        position * div_term
    )

    return pe


class FeedForward(nn.Module):

    def __init__(
        self,
        d_model: int,
        d_ff: int,
        dropout: float
    ):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):

        return self.net(x)


class Block(nn.Module):

    def __init__(self, cfg: LMConfig):

        super().__init__()

        self.ln1 = nn.LayerNorm(
            cfg.d_model
        )

        self.attn = CausalSelfAttention(
            cfg.d_model,
            cfg.n_heads,
            cfg.max_seq_len,
            cfg.dropout,
            attention_scale=cfg.attention_scale
        )

        self.ln2 = nn.LayerNorm(
            cfg.d_model
        )

        self.ffn = FeedForward(
            cfg.d_model,
            cfg.d_ff,
            cfg.dropout
        )

        self.norm_first = cfg.norm_first

    def forward(self, x):

        if self.norm_first:

            # Pre-norm
            x = x + self.attn(
                self.ln1(x)
            )

            x = x + self.ffn(
                self.ln2(x)
            )

        else:

            # Post-norm
            x = self.ln1(
                x + self.attn(x)
            )

            x = self.ln2(
                x + self.ffn(x)
            )

        return x


class DecoderLM(nn.Module):

    def __init__(self, cfg: LMConfig):

        super().__init__()

        self.cfg = cfg

        self.tok_emb = nn.Embedding(
            cfg.vocab_size,
            cfg.d_model
        )

        if cfg.pos_emb_type == "sinusoidal":

            pe = sinusoidal_positional_encoding(
                cfg.max_seq_len,
                cfg.d_model
            )

            self.register_buffer(
                "pos_emb",
                pe,
                persistent=False
            )

        self.blocks = nn.ModuleList(
            [
                Block(cfg)
                for _ in range(cfg.n_layers)
            ]
        )

        self.ln_f = nn.LayerNorm(
            cfg.d_model
        )

        self.head = nn.Linear(
            cfg.d_model,
            cfg.vocab_size,
            bias=False
        )

        # Weight tying
        self.head.weight = self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor):

        B, T = idx.shape

        if T > self.cfg.max_seq_len:

            raise ValueError(
                "Sequence length exceeds max_seq_len"
            )

        x = self.tok_emb(idx)

        if self.cfg.pos_emb_type == "sinusoidal":

            x = x + self.pos_emb[:T].unsqueeze(0)

        for block in self.blocks:

            x = block(x)

        x = self.ln_f(x)

        logits = self.head(x)

        return logits

Overwriting model_arch.py


In [34]:
import importlib, attention, model_arch
importlib.reload(attention)
importlib.reload(model_arch)
from model_arch import DecoderLM, LMConfig

In [15]:
import importlib
import sys

for name in ("model_arch", "attention"):
    sys.modules.pop(name, None)

attention = importlib.import_module("attention")
model_arch = importlib.import_module("model_arch")

LMConfig = model_arch.LMConfig
DecoderLM = model_arch.DecoderLM

print("Model code loaded.")

Model code loaded.


In [16]:
VOCAB_SIZE = tokenizer.vocab_size

MODEL_DIM = 560
MODEL_LAYERS = 8
MODEL_HEADS = 8
MODEL_FF = 2240
MODEL_SEQ_LEN = 2048

print("Vocabulary size:", VOCAB_SIZE)

Vocabulary size: 25000


In [10]:
from datasets import load_dataset

acts_dataset = load_dataset(
    "json",
    data_files=f"hf://datasets/Exploration-Lab/CS-781-Capstone/Phase_1_Pretraining/Train/train_acts.json",
    split="train"
)

Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['title', 'category', 'year', 'url', 'text', 'word_count', 'text_length_chars'],
    num_rows: 18826
})
{'title': 'Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996', 'category': 'Greater Bengaluru City Corporation - Act', 'year': 1972, 'url': 'https://indiankanoon.org/doc/179573747/', 'text': 'Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors including ground floor. The Corporation of the City of Bangalore granted permission for construction of buildings beyond five floors in certain cases. In respect of some of the buildings the permission granted by the Corporation contrary to the Zonal Regulations then in force was questioned in the High Court and the Division Bench of the High Court 

In [13]:
record = next(iter(acts_dataset))

start = time.time()

tokens = tokenizer.encode(record["text"])

print("Original text:")
print(record["text"][:500])

print("\nNumber of tokens:", len(tokens))
print("Encoding time:", round(time.time() - start, 3), "seconds")

print("\nDecoded text:")
print(tokenizer.decode(tokens)[:500])

Original text:
Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors including ground floor. The Corporation of the City of Bangalore granted permission for construction of buildings beyond five floors in certain cases. In respect of some of the buildings the permission gra

Number of tokens: 1503
Encoding time: 0.007 seconds

Decoded text:
bangalore city planning area zonal regulations (amendment and validation) act, 1996 (karnataka act no. 2 of 1996) statements of objects and reasons - act 2 of 1996. - the bangalore city planning area zonal regulations 1972, which was in vogue till 1984, provided for construction of five floors including ground floor. the corporation of the city of bangalore granted permission for construction of buildings beyond 

In [11]:
from datasets import load_dataset
import numpy as np
import random
import time

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

acts_dataset = load_dataset(
    "json",
    data_files={
        "train": f"hf://datasets/{HF_DATASET_REPO}/Phase_1_Pretraining/Train/train_acts.json"
    },
    split="train",
    streaming=True
)

print("Acts dataset loaded.")

Acts dataset loaded.


In [17]:
SEED = 42
TRAIN_TOKENS = 100_000_000
VAL_TOKENS = 5_000_000

TRAIN_PATH = "/kaggle/working/train.bin"
VAL_PATH = "/kaggle/working/val.bin"

train_count = 0
val_count = 0

random.seed(SEED)

start = time.time()

with open(TRAIN_PATH, "wb") as train_file, \
     open(VAL_PATH, "wb") as val_file:

    for data in acts_dataset:

        text = data["text"]
        tokens = tokenizer.encode(text, add_eos=True)

        if random.random() < 0.05:

            if val_count < VAL_TOKENS:

                remaining = VAL_TOKENS - val_count
                selected = tokens[:remaining]

                np.asarray(selected, dtype=np.uint16).tofile(val_file)
                val_count += len(selected)

        else:

            if train_count < TRAIN_TOKENS:

                remaining = TRAIN_TOKENS - train_count
                selected = tokens[:remaining]

                np.asarray(selected, dtype=np.uint16).tofile(train_file)
                train_count += len(selected)

        if (train_count + val_count) % 1_000_000 < len(tokens):
            print(
                f"Train: {train_count:,} | "
                f"Validation: {val_count:,} | "
                f"Time: {(time.time()-start)/60:.1f} min"
            )

        if train_count >= TRAIN_TOKENS and val_count >= VAL_TOKENS:
            break

print("\nFinished!")
print(f"Train tokens: {train_count:,}")
print(f"Validation tokens: {val_count:,}")

Train: 937,850 | Validation: 63,397 | Time: 0.3 min
Train: 1,958,689 | Validation: 63,397 | Time: 0.3 min
Train: 2,917,563 | Validation: 82,864 | Time: 0.3 min
Train: 3,896,421 | Validation: 105,680 | Time: 0.3 min
Train: 4,925,171 | Validation: 136,289 | Time: 0.3 min
Train: 5,838,113 | Validation: 162,813 | Time: 0.3 min
Train: 6,821,517 | Validation: 187,440 | Time: 0.3 min
Train: 7,799,395 | Validation: 205,082 | Time: 0.3 min
Train: 8,716,127 | Validation: 305,963 | Time: 0.4 min
Train: 9,670,436 | Validation: 331,062 | Time: 0.4 min
Train: 10,633,351 | Validation: 377,940 | Time: 0.4 min
Train: 11,607,866 | Validation: 397,623 | Time: 0.4 min
Train: 12,660,113 | Validation: 465,817 | Time: 0.4 min
Train: 13,534,522 | Validation: 466,766 | Time: 0.4 min
Train: 14,510,901 | Validation: 495,852 | Time: 0.4 min
Train: 15,372,574 | Validation: 628,184 | Time: 0.4 min
Train: 16,393,728 | Validation: 649,365 | Time: 0.4 min
Train: 17,296,157 | Validation: 750,813 | Time: 0.5 min
Train: 

In [21]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".bin"):
            print(os.path.join(root, file))

/kaggle/working/val.bin
/kaggle/working/train.bin


In [17]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

TRAIN_PATH = "/kaggle/input/models/strugglingug/model6/pytorch/default/1/train_v3.bin"
VAL_PATH = "/kaggle/input/models/strugglingug/model6/pytorch/default/1/val_v3.bin"

SEQ_LEN = 1024
BATCH_SIZE = 2


class TokenDataset(Dataset):

    def __init__(self, path, seq_len):
        self.tokens = np.memmap(
            path,
            dtype=np.uint16,
            mode="r"
        )
        self.seq_len = seq_len

    def __len__(self):
        return len(self.tokens) - self.seq_len

    def __getitem__(self, idx):

        x = torch.tensor(
            self.tokens[idx:idx + self.seq_len],
            dtype=torch.long
        )

        y = torch.tensor(
            self.tokens[idx + 1:idx + self.seq_len + 1],
            dtype=torch.long
        )

        return x, y


train_dataset = TokenDataset(TRAIN_PATH, SEQ_LEN)
val_dataset = TokenDataset(VAL_PATH, SEQ_LEN)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

from torch.utils.data import Subset
val_idx = np.random.default_rng(0).integers(0, len(val_dataset), size=40).tolist()
val_loader = DataLoader(Subset(val_dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False)

print("Train tokens:", len(train_dataset))
print("Validation tokens:", len(val_dataset))

Train tokens: 99998976
Validation tokens: 4998976


In [18]:
import torch

VOCAB_SIZE = tokenizer.vocab_size

MODEL_DIM = 768
MODEL_LAYERS = 12
MODEL_HEADS = 12
MODEL_FF = 2816

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

cfg = LMConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_DIM,
    n_layers=MODEL_LAYERS,
    n_heads=MODEL_HEADS,
    n_kv_heads=MODEL_HEADS,
    d_ff=MODEL_FF,
    max_seq_len=SEQ_LEN,
    dropout=0.1,
    pos_emb_type="sinusoidal",
    norm_first=True,
)

model = DecoderLM(cfg).to(device)

print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Device:", device)

Parameters: 99497472
Device: cuda


In [19]:
import time, os, math
import torch
import torch.nn.functional as F

TARGET_TOKENS = 100_000_000      # or len(train_dataset) to see your data about once
TRAIN_STEPS = TARGET_TOKENS // (BATCH_SIZE * SEQ_LEN)
WARMUP_STEPS = 200
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
USE_AMP = True                   # fp16 speed-up; set False if the loss becomes NaN
V = tokenizer.vocab_size

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
train_iter = iter(train_loader)

def lr_at(step):
    if step <= WARMUP_STEPS:
        return LEARNING_RATE * step / WARMUP_STEPS
    p = (step - WARMUP_STEPS) / max(1, TRAIN_STEPS - WARMUP_STEPS)
    return LEARNING_RATE * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * p)))

def next_batch(it, loader):
    try:
        return next(it), it
    except StopIteration:
        it = iter(loader)
        return next(it), it

def save_ckpt(path, step):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({"model_state": model.state_dict(), "config": vars(cfg),
                "vocab_size": V, "step": step}, path)

model.train()
t0 = time.time()
for step in range(1, TRAIN_STEPS + 1):
    for g in optimizer.param_groups:
        g["lr"] = lr_at(step)

    (x, y), train_iter = next_batch(train_iter, train_loader)
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
        logits = model(x)
    train_loss = F.cross_entropy(logits.float().reshape(-1, V), y.reshape(-1))
    scaler.scale(train_loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()

    if step == 1:
        print(f"first loss: {train_loss.item():.3f} (expect about 10.1)")

    if step % 100 == 0:
        model.eval()
        with torch.no_grad():
            losses = []
            for xv, yv in val_loader:
                xv, yv = xv.to(device), yv.to(device)
                with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                    vl = model(xv)
                losses.append(F.cross_entropy(vl.float().reshape(-1, V), yv.reshape(-1)).item())
        model.train()
        print(f"Step {step}/{TRAIN_STEPS} | Train {train_loss.item():.4f} | "
              f"Val {sum(losses)/len(losses):.4f} | {(time.time()-t0)/60:.1f} min")

    if step % 5000 == 0:
        save_ckpt("/kaggle/working/checkpoints/last.pt", step)

save_ckpt("/kaggle/working/checkpoints/decoder_final.pt", TRAIN_STEPS)

first loss: 10.276 (expect about 10.1)
Step 100/48828 | Train 8.0098 | Val 7.0326 | 0.7 min
Step 200/48828 | Train 6.3357 | Val 6.5619 | 1.1 min
Step 300/48828 | Train 6.4564 | Val 6.5234 | 1.6 min
Step 400/48828 | Train 7.0291 | Val 6.5343 | 2.1 min
Step 500/48828 | Train 6.0464 | Val 6.5124 | 2.5 min
Step 600/48828 | Train 6.2692 | Val 6.5183 | 3.0 min
Step 700/48828 | Train 6.7123 | Val 6.5191 | 3.5 min
Step 800/48828 | Train 6.4621 | Val 6.5150 | 3.9 min
Step 900/48828 | Train 7.0410 | Val 6.5030 | 4.4 min
Step 1000/48828 | Train 6.5970 | Val 6.4963 | 4.9 min
Step 1100/48828 | Train 6.7380 | Val 6.5012 | 5.3 min
Step 1200/48828 | Train 6.4344 | Val 6.4927 | 5.8 min
Step 1300/48828 | Train 6.1881 | Val 6.4971 | 6.3 min
Step 1400/48828 | Train 6.2089 | Val 6.4216 | 6.8 min
Step 1500/48828 | Train 6.3149 | Val 6.3197 | 7.2 min
Step 1600/48828 | Train 5.8429 | Val 6.1426 | 7.7 min
Step 1700/48828 | Train 5.4887 | Val 6.0512 | 8.2 min
Step 1800/48828 | Train 5.7446 | Val 6.0653 | 8.6 mi

In [24]:
from IPython.display import FileLink, display

display(FileLink(
    "checkpoints/decoder_final.pt",
    result_html_prefix="Download checkpoint: "
))

/kaggle/working/checkpoints/decoder_final.pt

In [15]:
def generate_n_words(model, tokenizer, prompt, n, device):

    model.eval()

    prompt_ids = tokenizer.encode(prompt)

    ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=device
    )

    generated_ids = []

    with torch.no_grad():

        while True:

            logits = model(ids)

            next_token_id = torch.argmax(
                logits[:, -1, :]
            ).item()

            generated_ids.append(next_token_id)

            ids = torch.cat([
                ids,
                torch.tensor(
                    [[next_token_id]],
                    dtype=torch.long,
                    device=device
                )
            ], dim=1)

            generated_text = tokenizer.decode(generated_ids)
            words = generated_text.split()

            if len(words) >= n:
                return " ".join(words[:n])

            if ids.shape[1] >= model.cfg.max_seq_len:
                return " ".join(words)

In [26]:
import json
from huggingface_hub import hf_hub_download, HfApi

REPO = "Exploration-Lab/CS-781-Capstone"
files = [f for f in HfApi().list_repo_files(REPO, repo_type="dataset") if "Phase_1" in f and "Test" in f]
print(files)                                   # check this is the Phase 1 test file
test_data = json.load(open(hf_hub_download(REPO, files[0], repo_type="dataset"), encoding="utf-8"))
print(len(test_data), "prompts")

['Phase_1_Pretraining/Test/test.json']


test.json: 0.00B [00:00, ?B/s]

1000 prompts


In [ ]:
block = cfg.max_seq_len        # 1024

@torch.no_grad()
def generate_batch(model, x, block, eos_id, temperature=0.8, top_k=50, rep_penalty=1.1, max_new_tokens=400):
    model.eval()
    B, T0 = x.shape
    for i in range(max_new_tokens):
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x[:, -block:])
        logits = (out[0] if isinstance(out, tuple) else out)[:, -1, :].float()
        logits[:, eos_id] = -float("inf")                  # we need a full 128 words
        window = x[:, -256:]
        s = logits.gather(1, window)
        s = torch.where(s > 0, s / rep_penalty, s * rep_penalty)
        logits.scatter_(1, window, s)
        logits = logits / temperature
        kth = torch.topk(logits, top_k)[0][:, -1:]
        logits[logits < kth] = -float("inf")
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
        x = torch.cat([x, nxt], dim=1)
        if (i + 1) % 16 == 0 and all(len(tokenizer.decode(r[T0:].tolist()).split()) > 128 for r in x):
            break
    return [tokenizer.decode(r[T0:].tolist()) for r in x]

In [31]:
def generate_texts(prompts, batch=8, prompt_tokens=800, **gen_kw):
    ids = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(ids)), key=lambda i: len(ids[i]))
    outs = [None] * len(ids)
    for s in range(0, len(order), batch):
        idx = order[s:s + batch]
        L = min(prompt_tokens, block, min(len(ids[i]) for i in idx))
        x = torch.tensor([ids[i][-L:] for i in idx], device=device)
        for i, t in zip(idx, generate_batch(model, x, block, tokenizer.eos_id, **gen_kw)):
            outs[i] = " ".join(t.split()[:128])
    return outs

In [33]:
from tqdm.auto import tqdm

SMOKE = False
PROMPT_TOKENS = 800
BATCH = 8
torch.manual_seed(42)

keys = list(test_data)[:8] if SMOKE else list(test_data)
ids = {k: tokenizer.encode(test_data[k]["text"]) for k in keys}
keys = sorted(keys, key=lambda k: len(ids[k]))          # similar lengths share a batch

predictions = {}
for i in tqdm(range(0, len(keys), BATCH)):
    kb = keys[i:i + BATCH]
    L = min(PROMPT_TOKENS, block, min(len(ids[k]) for k in kb))
    x = torch.tensor([ids[k][-L:] for k in kb], device=device)
    for k, t in zip(kb, generate_batch(model, x, block, tokenizer.eos_id)):
        predictions[k] = {"text": " ".join(t.split()[:128])}

if SMOKE:
    for k in keys[:2]:
        print(predictions[k]["text"], "\n---")

  0%|          | 0/125 [00:00<?, ?it/s]

In [38]:
import zipfile
TAG = "T0.8_k50"          # change per run so you keep separate zips
assert not SMOKE, "set SMOKE = False and re-run the generation cell first"
assert set(predictions) == set(test_data)
bad = [k for k, v in predictions.items() if len(v["text"].split()) != 128]
assert not bad, f"{len(bad)} predictions are not exactly 128 words, e.g. {bad[:3]}"

with open("prediction.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False)
with zipfile.ZipFile(f"submission_{TAG}.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("prediction.json", arcname="prediction.json")
print(zipfile.ZipFile(f"submission_{TAG}.zip").namelist())     # must print ['prediction.json']

['prediction.json']


In [37]:
k = "803"

L = min(PROMPT_TOKENS, block, len(ids[k]))
x = torch.tensor([ids[k][-L:]], device=device)

t = generate_batch(
    model, x, block, tokenizer.eos_id,
    temperature=0.8,
    top_k=50,
    rep_penalty=1.1,
    max_new_tokens=700
)[0]

predictions[k] = {"text": " ".join(t.split()[:128])}

print("Word count:", len(predictions[k]["text"].split()))
print(predictions[k]["text"])

Word count: 128
that the same is on phytosanitarys under the existing scheme of 21 years, 19%. it has been declared to pay the tax recovery within 10 per cent of the month year. 8. the amount so charged should be as follows: - i. on the above period, an amount of three years will be made out of each week should be credited in the case of a company, bank, bank will be shown in the case of a company, bank, trust will be provided into the treasury instalments. 3. however, the following amount will be deducted from the account office from the government of india. 2. the said cost will be fixed in the office to the bank as per the terms for which the amount was issued: 1.


In [6]:
%%writefile attention_ed.py

"""Multi-head attention (bidirectional / causal / cross) with RoPE and QK-norm, from raw tensor ops."""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):

    def __init__(self, dim: int, eps: float = 1e-6):

        super().__init__()

        self.eps = eps

        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        x_f = x.float()

        x_f = x_f * torch.rsqrt(x_f.pow(2).mean(-1, keepdim=True) + self.eps)

        return (x_f * self.weight).to(x.dtype)


def build_rope_cache(max_len: int, d_head: int, base: float = 10000.0):

    # one rotation frequency per pair of channels
    inv_freq = 1.0 / (
        base ** (torch.arange(0, d_head, 2, dtype=torch.float32) / d_head)
    )

    positions = torch.arange(max_len, dtype=torch.float32)

    angles = torch.outer(positions, inv_freq)       # (max_len, d_head/2)

    return torch.cos(angles), torch.sin(angles)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:

    # x: (B, n_heads, T, d_head)   cos, sin: (T, d_head/2)
    # rotate every channel pair (x1, x2) by an angle that grows with position

    x_f = x.float()

    x1 = x_f[..., 0::2]
    x2 = x_f[..., 1::2]

    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]

    out = torch.stack(
        (x1 * cos - x2 * sin, x1 * sin + x2 * cos),
        dim=-1
    ).flatten(-2)

    return out.to(x.dtype)


class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        max_q_len: int,
        max_kv_len: int,
        dropout: float = 0.0,
        causal: bool = False,
        attention_scale=None,
        rope: bool = False,
        qk_norm: bool = True
    ):

        super().__init__()

        assert d_model % n_heads == 0

        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.causal = causal
        self.rope = rope
        self.qk_norm = qk_norm

        self.attention_scale = (
            1.0 / math.sqrt(self.d_head)
            if attention_scale is None
            else float(attention_scale)
        )

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        if qk_norm:

            # normalise every query / key vector (per head) to unit RMS with a learned gain:
            # attention logits stay bounded, so training stays stable at higher learning rates
            self.q_norm = RMSNorm(self.d_head)
            self.k_norm = RMSNorm(self.d_head)

        if causal:

            causal_mask = torch.tril(
                torch.ones(
                    max_q_len,
                    max_kv_len,
                    dtype=torch.bool
                )
            )

            self.register_buffer(
                "causal_mask",
                causal_mask,
                persistent=False
            )

        if rope:

            # rotary positions only make sense for SELF-attention (queries and keys come
            # from the same sequence); cross-attention is built with rope=False
            assert self.d_head % 2 == 0

            cos, sin = build_rope_cache(
                max(max_q_len, max_kv_len),
                self.d_head
            )

            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x: torch.Tensor, context: torch.Tensor = None) -> torch.Tensor:

        # x       : (B, Tq, C)   -> queries
        # context : (B, Tk, C)   -> keys / values  (None = self-attention)

        B, Tq, C = x.shape

        kv = x if context is None else context

        Tk = kv.shape[1]

        q = self.q_proj(x)
        k = self.k_proj(kv)
        v = self.v_proj(kv)

        q = q.view(B, Tq, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)

        if self.qk_norm:

            q = self.q_norm(q)
            k = self.k_norm(k)

        if self.rope:

            q = apply_rope(q, self.rope_cos[:Tq], self.rope_sin[:Tq])
            k = apply_rope(k, self.rope_cos[:Tk], self.rope_sin[:Tk])

        q = q * self.attention_scale

        # (B, n_heads, Tq, Tk)
        scores = torch.matmul(q, k.transpose(-2, -1))

        if self.causal:

            mask = self.causal_mask[:Tq, :Tk]

            scores = scores.masked_fill(~mask, float("-inf"))

        attn = F.softmax(scores, dim=-1)

        attn = self.attn_dropout(attn)

        out = torch.matmul(attn, v)

        out = out.transpose(1, 2).contiguous().view(B, Tq, C)

        out = self.out_proj(out)

        out = self.resid_dropout(out)

        return out

Overwriting attention_ed.py


In [7]:
%%writefile model_arch_ed.py

"""Encoder-Decoder Transformer LM (RoPE + QK-norm), built from scratch."""

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

from attention_ed import MultiHeadAttention


@dataclass
class Seq2SeqConfig:

    vocab_size: int

    d_model: int = 768
    n_enc_layers: int = 6
    n_dec_layers: int = 4
    n_heads: int = 12

    d_ff: int = 3072

    max_src_len: int = 512
    max_tgt_len: int = 512

    dropout: float = 0.0

    pos_emb_type: str = "rope"          # "rope" (rotary, relative) or "sinusoidal" (absolute)

    attention_scale: float = None

    norm_first: bool = True

    qk_norm: bool = True


def sinusoidal_positional_encoding(
    max_len: int,
    d_model: int
) -> torch.Tensor:

    pe = torch.zeros(max_len, d_model)

    position = torch.arange(
        max_len,
        dtype=torch.float32
    ).unsqueeze(1)

    div_term = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float32)
        * (-math.log(10000.0) / d_model)
    )

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    return pe


class FeedForward(nn.Module):

    def __init__(self, d_model: int, d_ff: int, dropout: float):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):

        return self.net(x)


class EncoderBlock(nn.Module):

    def __init__(self, cfg: Seq2SeqConfig):

        super().__init__()

        self.ln1 = nn.LayerNorm(cfg.d_model)

        # bidirectional self-attention (no mask)
        self.attn = MultiHeadAttention(
            cfg.d_model,
            cfg.n_heads,
            cfg.max_src_len,
            cfg.max_src_len,
            cfg.dropout,
            causal=False,
            attention_scale=cfg.attention_scale,
            rope=(cfg.pos_emb_type == "rope"),
            qk_norm=cfg.qk_norm
        )

        self.ln2 = nn.LayerNorm(cfg.d_model)

        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

        self.norm_first = cfg.norm_first

    def forward(self, x):

        if self.norm_first:

            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))

        else:

            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))

        return x


class DecoderBlock(nn.Module):

    def __init__(self, cfg: Seq2SeqConfig):

        super().__init__()

        self.ln1 = nn.LayerNorm(cfg.d_model)

        # causal self-attention over the target tokens
        self.self_attn = MultiHeadAttention(
            cfg.d_model,
            cfg.n_heads,
            cfg.max_tgt_len,
            cfg.max_tgt_len,
            cfg.dropout,
            causal=True,
            attention_scale=cfg.attention_scale,
            rope=(cfg.pos_emb_type == "rope"),
            qk_norm=cfg.qk_norm
        )

        self.ln2 = nn.LayerNorm(cfg.d_model)

        # cross-attention: queries = decoder, keys/values = encoder output (no RoPE here)
        self.cross_attn = MultiHeadAttention(
            cfg.d_model,
            cfg.n_heads,
            cfg.max_tgt_len,
            cfg.max_src_len,
            cfg.dropout,
            causal=False,
            attention_scale=cfg.attention_scale,
            rope=False,
            qk_norm=cfg.qk_norm
        )

        self.ln3 = nn.LayerNorm(cfg.d_model)

        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

        self.norm_first = cfg.norm_first

    def forward(self, x, enc_out):

        if self.norm_first:

            x = x + self.self_attn(self.ln1(x))
            x = x + self.cross_attn(self.ln2(x), enc_out)
            x = x + self.ffn(self.ln3(x))

        else:

            x = self.ln1(x + self.self_attn(x))
            x = self.ln2(x + self.cross_attn(x, enc_out))
            x = self.ln3(x + self.ffn(x))

        return x


class EncoderDecoderLM(nn.Module):

    def __init__(self, cfg: Seq2SeqConfig):

        super().__init__()

        self.cfg = cfg

        # one embedding table shared by the encoder, the decoder and the output head
        self.tok_emb = nn.Embedding(
            cfg.vocab_size,
            cfg.d_model
        )

        if cfg.pos_emb_type == "sinusoidal":

            pe = sinusoidal_positional_encoding(
                max(cfg.max_src_len, cfg.max_tgt_len),
                cfg.d_model
            )

            self.register_buffer(
                "pos_emb",
                pe,
                persistent=False
            )

        self.enc_blocks = nn.ModuleList(
            [EncoderBlock(cfg) for _ in range(cfg.n_enc_layers)]
        )

        self.enc_ln = nn.LayerNorm(cfg.d_model)

        self.dec_blocks = nn.ModuleList(
            [DecoderBlock(cfg) for _ in range(cfg.n_dec_layers)]
        )

        self.dec_ln = nn.LayerNorm(cfg.d_model)

        self.head = nn.Linear(
            cfg.d_model,
            cfg.vocab_size,
            bias=False
        )

        # Weight tying
        self.head.weight = self.tok_emb.weight

        # small embedding init (a N(0,1) tied embedding gives a huge first loss)
        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=0.02)

    def encode(self, src: torch.Tensor) -> torch.Tensor:

        B, S = src.shape

        if S > self.cfg.max_src_len:

            raise ValueError("Source length exceeds max_src_len")

        x = self.tok_emb(src)

        if self.cfg.pos_emb_type == "sinusoidal":

            x = x + self.pos_emb[:S].unsqueeze(0)

        for block in self.enc_blocks:

            x = block(x)

        return self.enc_ln(x)

    def decode(self, tgt_in: torch.Tensor, enc_out: torch.Tensor) -> torch.Tensor:

        B, T = tgt_in.shape

        if T > self.cfg.max_tgt_len:

            raise ValueError("Target length exceeds max_tgt_len")

        x = self.tok_emb(tgt_in)

        if self.cfg.pos_emb_type == "sinusoidal":

            x = x + self.pos_emb[:T].unsqueeze(0)

        for block in self.dec_blocks:

            x = block(x, enc_out)

        x = self.dec_ln(x)

        logits = self.head(x)

        return logits

    def forward(self, src: torch.Tensor, tgt_in: torch.Tensor):

        return self.decode(tgt_in, self.encode(src))

Overwriting model_arch_ed.py


In [27]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset

TRAIN_PATH = "/kaggle/input/models/strugglingug/model6/pytorch/default/1/train_v3.bin"
VAL_PATH = "/kaggle/input/models/strugglingug/model6/pytorch/default/1/val_v3.bin"

SRC_LEN = 512
TGT_LEN = 512
BATCH_SIZE = 4
BOS_ID = tokenizer.eos_id


class Seq2SeqDataset(Dataset):

    def __init__(self, path, src_len, tgt_len, bos_id):
        self.tokens = np.memmap(path, dtype=np.uint16, mode="r")
        self.src_len = src_len
        self.tgt_len = tgt_len
        self.bos_id = bos_id

    def __len__(self):
        return len(self.tokens) - self.src_len - self.tgt_len

    def __getitem__(self, idx):
        s = np.asarray(self.tokens[idx:idx + self.src_len], dtype=np.int64)
        t = np.asarray(self.tokens[idx + self.src_len:idx + self.src_len + self.tgt_len], dtype=np.int64)
        src = torch.from_numpy(s)
        tgt = torch.from_numpy(t)
        tgt_in = torch.cat([torch.tensor([self.bos_id], dtype=torch.long), tgt[:-1]])
        return src, tgt_in, tgt


train_dataset = Seq2SeqDataset(TRAIN_PATH, SRC_LEN, TGT_LEN, BOS_ID)
val_dataset = Seq2SeqDataset(VAL_PATH, SRC_LEN, TGT_LEN, BOS_ID)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_idx = np.random.default_rng(0).integers(0, len(val_dataset), size=40).tolist()
val_loader = DataLoader(Subset(val_dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False)

print("Train windows:", len(train_dataset))
print("Validation windows:", len(val_dataset))

Train windows: 145028515
Validation windows: 4969437


In [32]:
import importlib, attention_ed, model_arch_ed
importlib.reload(attention_ed)
importlib.reload(model_arch_ed)
from model_arch_ed import EncoderDecoderLM, Seq2SeqConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = Seq2SeqConfig(
    vocab_size=tokenizer.vocab_size,
    d_model=768,
    n_enc_layers=6,
    n_dec_layers=4,
    n_heads=12,
    d_ff=3072,
    max_src_len=SRC_LEN,
    max_tgt_len=TGT_LEN,
    dropout=0.0,                 # < 1 epoch of data: dropout only slows learning
    pos_emb_type="rope",
    qk_norm=True,
    norm_first=True
)

model = EncoderDecoderLM(cfg).to(device)

print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Device:", device)

Parameters: 103336192
Device: cuda


In [33]:
import time, math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

V = tokenizer.vocab_size
ACCUM_STEPS = 2
WEIGHT_DECAY = 0.1
BETAS = (0.9, 0.95)
GRAD_CLIP = 1.0
USE_AMP = True

RUN_SWEEP = False
SWEEP_LRS = [3e-4, 6e-4, 1e-3, 2e-3]
SWEEP_STEPS = 600                    # 600 steps x 8,192 tokens = about 5M tokens per trial
LR_SAFETY = 0.8                      # short sweeps favour slightly high LRs; use a bit less for the long run


def make_lr_fn(peak_lr, total_steps, warmup_steps, min_ratio=0.1):
    def fn(step):
        if step <= warmup_steps:
            return peak_lr * step / warmup_steps
        p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return peak_lr * (min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * p)))
    return fn


@torch.no_grad()
def evaluate(m):
    m.eval()
    losses = []
    for sv, tiv, tv in val_loader:
        sv, tiv, tv = sv.to(device), tiv.to(device), tv.to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            vl = m(sv, tiv)
        losses.append(F.cross_entropy(vl.float().reshape(-1, V), tv.reshape(-1)).item())
        del vl
    m.train()
    return sum(losses) / len(losses)


def sweep_trial(lr, steps):
    torch.manual_seed(0)
    m = EncoderDecoderLM(cfg).to(device)
    m.train()
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=WEIGHT_DECAY, betas=BETAS)
    scl = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    lr_fn = make_lr_fn(lr, steps, max(20, int(0.05 * steps)))
    loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(0))      # same batches for every LR
    it = iter(loader)
    val = float("nan")
    for step in range(1, steps + 1):
        for g in opt.param_groups:
            g["lr"] = lr_fn(step)
        opt.zero_grad(set_to_none=True)
        total = 0.0
        for _ in range(ACCUM_STEPS):
            src, tgt_in, tgt = next(it)
            src, tgt_in, tgt = src.to(device), tgt_in.to(device), tgt.to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = m(src, tgt_in)
            loss = F.cross_entropy(logits.float().reshape(-1, V), tgt.reshape(-1)) / ACCUM_STEPS
            scl.scale(loss).backward()
            total += loss.item()
            del logits, loss
        if not math.isfinite(total):
            print(f"  lr {lr:g}: loss became NaN/inf at step {step} -> diverged")
            break
        scl.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(m.parameters(), GRAD_CLIP)
        scl.step(opt)
        scl.update()
    else:
        val = evaluate(m)
    del m, opt
    torch.cuda.empty_cache()
    return val


if RUN_SWEEP:
    results = {}
    for lr in SWEEP_LRS:
        t = time.time()
        results[lr] = sweep_trial(lr, SWEEP_STEPS)
        print(f"lr {lr:g} -> val loss {results[lr]:.4f}  ({(time.time() - t) / 60:.1f} min)")
    valid = {k: v for k, v in results.items() if math.isfinite(v)}
    best_lr = min(valid, key=valid.get)
    LEARNING_RATE = best_lr * LR_SAFETY
    print(f"\nbest sweep LR = {best_lr:g}  ->  using LEARNING_RATE = {LEARNING_RATE:.2e} for the main run")
else:
    LEARNING_RATE = 6e-4
    print("sweep skipped, LEARNING_RATE =", LEARNING_RATE)

sweep skipped, LEARNING_RATE = 0.0006


In [34]:
import time, math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

V = tokenizer.vocab_size
ACCUM_STEPS = 2
WEIGHT_DECAY = 0.1
BETAS = (0.9, 0.95)
GRAD_CLIP = 1.0
USE_AMP = True
LEARNING_RATE = 1.6e-3

In [35]:
import time, os, math
import torch
import torch.nn.functional as F

TARGET_TOKENS = 145_000_000          # tokens consumed (source + target); raise if time allows
TRAIN_STEPS = TARGET_TOKENS // (BATCH_SIZE * ACCUM_STEPS * (SRC_LEN + TGT_LEN))
WARMUP_STEPS = 300
MIN_LR_RATIO = 0.1
CKPT_DIR = "/kaggle/working/checkpoints_encdec"

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=BETAS)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
train_iter = iter(train_loader)


def lr_at(step):
    if step <= WARMUP_STEPS:
        return LEARNING_RATE * step / WARMUP_STEPS
    p = (step - WARMUP_STEPS) / max(1, TRAIN_STEPS - WARMUP_STEPS)
    return LEARNING_RATE * (MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * p)))


def next_batch(it, loader):
    try:
        return next(it), it
    except StopIteration:
        it = iter(loader)
        return next(it), it


def save_ckpt(path, step):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({"model_state": model.state_dict(), "config": vars(cfg), "vocab_size": V, "step": step,
                "optimizer_state": optimizer.state_dict(), "scaler_state": scaler.state_dict()}, path)


RESUME_FROM = None                  # after a crash: "/kaggle/.../last.pt"
start_step = 0
if RESUME_FROM:
    ckpt = torch.load(RESUME_FROM, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    start_step = ckpt["step"]
    if "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scaler.load_state_dict(ckpt["scaler_state"])
    print("resumed from step", start_step)

print(f"tokens consumed/step: {BATCH_SIZE * ACCUM_STEPS * (SRC_LEN + TGT_LEN):,} | steps: {TRAIN_STEPS:,} | peak LR: {LEARNING_RATE:.2e}")
model.train()
t0 = time.time()
for step in range(start_step + 1, TRAIN_STEPS + 1):
    for g in optimizer.param_groups:
        g["lr"] = lr_at(step)

    optimizer.zero_grad(set_to_none=True)
    train_loss = 0.0
    for _ in range(ACCUM_STEPS):
        (src, tgt_in, tgt), train_iter = next_batch(train_iter, train_loader)
        src, tgt_in, tgt = src.to(device), tgt_in.to(device), tgt.to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(src, tgt_in)
        loss = F.cross_entropy(logits.float().reshape(-1, V), tgt.reshape(-1)) / ACCUM_STEPS
        scaler.scale(loss).backward()
        train_loss += loss.item()
        del logits, loss

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()

    if step == start_step + 1:
        print(f"first loss: {train_loss:.3f} (expect about 10.1-10.6) | "
              f"peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

    if step % 100 == 0:
        val_loss = evaluate(model)
        print(f"Step {step}/{TRAIN_STEPS} | Train {train_loss:.4f} | Val {val_loss:.4f} | "
              f"LR {lr_at(step):.2e} | {(time.time()-t0)/60:.1f} min")

    if step % 3000 == 0:
        save_ckpt(f"{CKPT_DIR}/last.pt", step)

save_ckpt(f"{CKPT_DIR}/encdec_final1.pt", TRAIN_STEPS)

tokens consumed/step: 8,192 | steps: 17,700 | peak LR: 1.60e-03
first loss: 10.464 (expect about 10.1-10.6) | peak GPU memory: 6.7 GB
Step 100/17700 | Train 5.9342 | Val 6.2353 | LR 5.33e-04 | 1.2 min
Step 200/17700 | Train 5.5986 | Val 5.5909 | LR 1.07e-03 | 2.0 min
Step 300/17700 | Train 4.8608 | Val 5.2445 | LR 1.60e-03 | 2.8 min
Step 400/17700 | Train 4.8565 | Val 5.0197 | LR 1.60e-03 | 3.6 min
Step 500/17700 | Train 4.5311 | Val 4.8666 | LR 1.60e-03 | 4.4 min
Step 600/17700 | Train 4.6178 | Val 4.6998 | LR 1.60e-03 | 5.2 min
Step 700/17700 | Train 4.5020 | Val 4.5875 | LR 1.60e-03 | 6.0 min
Step 800/17700 | Train 4.0417 | Val 4.4821 | LR 1.60e-03 | 6.9 min
Step 900/17700 | Train 3.8201 | Val 4.4398 | LR 1.60e-03 | 7.7 min
Step 1000/17700 | Train 4.1798 | Val 4.3488 | LR 1.59e-03 | 8.5 min
Step 1100/17700 | Train 4.1942 | Val 4.2805 | LR 1.59e-03 | 9.3 min
Step 1200/17700 | Train 3.8173 | Val 4.2308 | LR 1.59e-03 | 10.1 min
Step 1300/17700 | Train 4.3144 | Val 4.1813 | LR 1.59e-03 

In [36]:
import json
from huggingface_hub import hf_hub_download, HfApi

REPO = "Exploration-Lab/CS-781-Capstone"
files = [f for f in HfApi().list_repo_files(REPO, repo_type="dataset") if "Phase_1" in f and "Test" in f]
print(files)                                   # check this is the Phase 1 test file
test_data = json.load(open(hf_hub_download(REPO, files[0], repo_type="dataset"), encoding="utf-8"))
print(len(test_data), "prompts")

['Phase_1_Pretraining/Test/test.json']


test.json: 0.00B [00:00, ?B/s]

1000 prompts


In [13]:
import torch.nn.functional as F

BOS_ID = tokenizer.eos_id


@torch.no_grad()
def generate_batch_ed(model, src, bos_id, eos_id, temperature=0.8, top_k=50, rep_penalty=1.1, max_new_tokens=400):
    model.eval()
    B = src.shape[0]
    max_new_tokens = min(max_new_tokens, model.cfg.max_tgt_len - 1)
    with torch.autocast("cuda", dtype=torch.float16):
        enc_out = model.encode(src)                     # the encoder runs ONCE per prompt
    y = torch.full((B, 1), bos_id, dtype=torch.long, device=src.device)
    for i in range(max_new_tokens):
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model.decode(y, enc_out)[:, -1, :].float()
        logits[:, eos_id] = -float("inf")               # we need a full 128 words
        window = y[:, -256:]
        s = logits.gather(1, window)
        s = torch.where(s > 0, s / rep_penalty, s * rep_penalty)
        logits.scatter_(1, window, s)
        logits = logits / temperature
        kth = torch.topk(logits, top_k)[0][:, -1:]
        logits[logits < kth] = -float("inf")
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
        y = torch.cat([y, nxt], dim=1)
        if (i + 1) % 16 == 0 and all(len(tokenizer.decode(r[1:].tolist()).split()) > 128 for r in y):
            break
    return [tokenizer.decode(r[1:].tolist()) for r in y]


from tqdm.auto import tqdm

SMOKE = True                 # True = 16 prompts only; set False for the full test set
BATCH = 16
torch.manual_seed(42)

keys = list(test_data)[:16] if SMOKE else list(test_data)
ids = {k: tokenizer.encode(test_data[k]["text"]) for k in keys}
L = min(cfg.max_src_len, min(len(v) for v in ids.values()))
print("source length used:", L, "tokens")

predictions = {}
for i in tqdm(range(0, len(keys), BATCH)):
    kb = keys[i:i + BATCH]
    src = torch.tensor([ids[k][-L:] for k in kb], device=device)
    for k, t in zip(kb, generate_batch_ed(model, src, BOS_ID, tokenizer.eos_id)):
        predictions[k] = {"text": " ".join(t.split()[:128])}

if SMOKE:
    for k in keys[:2]:
        print(predictions[k]["text"], "\n---")

source length used: 512 tokens


  0%|          | 0/1 [00:00<?, ?it/s]

any other document (other than an evidence) or any document relating to the same; or (c) when, a petition is presented as a petition for reference should not be accompanied by the following documents: 1. written statement of all papers; 2. copy of paper; and 3. copy of document.(other than document) 4. document for each paper. 5. list of papers which the paper can be produced in evidence. 6. copy of paper (see rule 3). 7. certified copies of paper. 8. list of papers in original paper. 9. copy of paper. 10. copy of paper-books. 11. copies of documents showing index of papers containing paper. 12. memorandum of appeal. 13. original paper. 14. documents filed under section 4. 15. documents filed under section 5 (6)(i). 16. appeal under 
---
1. 4. 5. 6. 7. 8. 10. 11. 12. 13. 14. schedule 7 (see rule 4) the annual report of the state of orissa state electricity board, assam board of revenue and revenue department, orissa. part i - general sl. no. | name | category of the state | name of the

In [37]:
import torch.nn.functional as F

BOS_ID = tokenizer.eos_id


@torch.no_grad()
def generate_batch_ed(model, src, bos_id, eos_id, temperature=0.8, top_k=50, rep_penalty=1.1, max_new_tokens=400):
    model.eval()
    B = src.shape[0]
    max_new_tokens = min(max_new_tokens, model.cfg.max_tgt_len - 1)
    with torch.autocast("cuda", dtype=torch.float16):
        enc_out = model.encode(src)                     # the encoder runs ONCE per prompt
    y = torch.full((B, 1), bos_id, dtype=torch.long, device=src.device)
    for i in range(max_new_tokens):
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model.decode(y, enc_out)[:, -1, :].float()
        logits[:, eos_id] = -float("inf")               # we need a full 128 words
        window = y[:, -256:]
        s = logits.gather(1, window)
        s = torch.where(s > 0, s / rep_penalty, s * rep_penalty)
        logits.scatter_(1, window, s)
        logits = logits / temperature
        kth = torch.topk(logits, top_k)[0][:, -1:]
        logits[logits < kth] = -float("inf")
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
        y = torch.cat([y, nxt], dim=1)
        if (i + 1) % 16 == 0 and all(len(tokenizer.decode(r[1:].tolist()).split()) > 128 for r in y):
            break
    return [tokenizer.decode(r[1:].tolist()) for r in y]


from tqdm.auto import tqdm

SMOKE = False                 # True = 16 prompts only; set False for the full test set
BATCH = 16
torch.manual_seed(42)

keys = list(test_data)[:16] if SMOKE else list(test_data)
ids = {k: tokenizer.encode(test_data[k]["text"]) for k in keys}
L = min(cfg.max_src_len, min(len(v) for v in ids.values()))
print("source length used:", L, "tokens")

predictions = {}
for i in tqdm(range(0, len(keys), BATCH)):
    kb = keys[i:i + BATCH]
    src = torch.tensor([ids[k][-L:] for k in kb], device=device)
    for k, t in zip(kb, generate_batch_ed(model, src, BOS_ID, tokenizer.eos_id)):
        predictions[k] = {"text": " ".join(t.split()[:128])}

if SMOKE:
    for k in keys[:2]:
        print(predictions[k]["text"], "\n---")

source length used: 512 tokens


  0%|          | 0/63 [00:00<?, ?it/s]

In [38]:
import zipfile

# 1) repair any prediction that came out shorter than 128 words
for attempt in range(5):
    bad = [k for k, v in predictions.items() if len(v["text"].split()) != 128]
    if not bad:
        break
    print(f"attempt {attempt + 1}: regenerating {len(bad)} prediction(s)")
    torch.manual_seed(1000 + attempt)
    for i in range(0, len(bad), BATCH):
        kb = bad[i:i + BATCH]
        src = torch.tensor([ids[k][-L:] for k in kb], device=device)
        for k, t in zip(kb, generate_batch_ed(model, src, BOS_ID, tokenizer.eos_id, max_new_tokens=500)):
            words = t.split()
            if len(words) >= 128:
                predictions[k] = {"text": " ".join(words[:128])}
            elif len(words) > len(predictions[k]["text"].split()):
                predictions[k] = {"text": " ".join(words)}
for k in [k for k, v in predictions.items() if len(v["text"].split()) != 128]:
    words = predictions[k]["text"].split()
    if words:                                               # last resort: repeat its own words
        predictions[k] = {"text": " ".join((words * (128 // len(words) + 1))[:128])}

# 2) validate, write, zip
TAG = "encdec_T0.8_k50"
assert not SMOKE, "set SMOKE = False and re-run the generation cell first"
assert set(predictions) == set(test_data)
bad = [k for k, v in predictions.items() if len(v["text"].split()) != 128]
assert not bad, f"{len(bad)} predictions are not exactly 128 words, e.g. {bad[:3]}"

with open("prediction.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False)
with zipfile.ZipFile(f"submission_{TAG}2.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("prediction.json", arcname="prediction.json")
print(zipfile.ZipFile(f"submission_{TAG}.zip").namelist())     # must print ['prediction.json']

attempt 1: regenerating 5 prediction(s)
['prediction.json']


In [1]:
print("hello")

hello


In [39]:
from IPython.display import FileLink, display

display(FileLink(
    "checkpoints_encdec/encdec_final1.pt",
    result_html_prefix="Download checkpoint: "
))

/kaggle/working/checkpoints_encdec/encdec_final1.pt

In [17]:
%%writefile attention_ed.py

"""Multi-head attention (bidirectional / causal / cross) with RoPE and QK-norm, from raw tensor ops."""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):

    def __init__(self, dim: int, eps: float = 1e-6):

        super().__init__()

        self.eps = eps

        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        x_f = x.float()

        x_f = x_f * torch.rsqrt(x_f.pow(2).mean(-1, keepdim=True) + self.eps)

        return (x_f * self.weight).to(x.dtype)


def build_rope_cache(max_len: int, d_head: int, base: float = 10000.0):

    # one rotation frequency per pair of channels
    inv_freq = 1.0 / (
        base ** (torch.arange(0, d_head, 2, dtype=torch.float32) / d_head)
    )

    positions = torch.arange(max_len, dtype=torch.float32)

    angles = torch.outer(positions, inv_freq)       # (max_len, d_head/2)

    return torch.cos(angles), torch.sin(angles)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:

    # x: (B, n_heads, T, d_head)   cos, sin: (T, d_head/2)
    # rotate every channel pair (x1, x2) by an angle that grows with position

    x_f = x.float()

    x1 = x_f[..., 0::2]
    x2 = x_f[..., 1::2]

    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]

    out = torch.stack(
        (x1 * cos - x2 * sin, x1 * sin + x2 * cos),
        dim=-1
    ).flatten(-2)

    return out.to(x.dtype)


class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        max_q_len: int,
        max_kv_len: int,
        dropout: float = 0.0,
        causal: bool = False,
        attention_scale=None,
        rope: bool = False,
        qk_norm: bool = True
    ):

        super().__init__()

        assert d_model % n_heads == 0

        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.causal = causal
        self.rope = rope
        self.qk_norm = qk_norm

        self.attention_scale = (
            1.0 / math.sqrt(self.d_head)
            if attention_scale is None
            else float(attention_scale)
        )

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        if qk_norm:

            # normalise every query / key vector (per head) to unit RMS with a learned gain:
            # attention logits stay bounded, so training stays stable at higher learning rates
            self.q_norm = RMSNorm(self.d_head)
            self.k_norm = RMSNorm(self.d_head)

        if causal:

            causal_mask = torch.tril(
                torch.ones(
                    max_q_len,
                    max_kv_len,
                    dtype=torch.bool
                )
            )

            self.register_buffer(
                "causal_mask",
                causal_mask,
                persistent=False
            )

        if rope:

            # rotary positions only make sense for SELF-attention (queries and keys come
            # from the same sequence); cross-attention is built with rope=False
            assert self.d_head % 2 == 0

            cos, sin = build_rope_cache(
                max(max_q_len, max_kv_len),
                self.d_head
            )

            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x: torch.Tensor, context: torch.Tensor = None) -> torch.Tensor:

        # x       : (B, Tq, C)   -> queries
        # context : (B, Tk, C)   -> keys / values  (None = self-attention)

        B, Tq, C = x.shape

        kv = x if context is None else context

        Tk = kv.shape[1]

        q = self.q_proj(x)
        k = self.k_proj(kv)
        v = self.v_proj(kv)

        q = q.view(B, Tq, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)

        if self.qk_norm:

            q = self.q_norm(q)
            k = self.k_norm(k)

        if self.rope:

            q = apply_rope(q, self.rope_cos[:Tq], self.rope_sin[:Tq])
            k = apply_rope(k, self.rope_cos[:Tk], self.rope_sin[:Tk])

        q = q * self.attention_scale

        # (B, n_heads, Tq, Tk)
        scores = torch.matmul(q, k.transpose(-2, -1))

        if self.causal:

            mask = self.causal_mask[:Tq, :Tk]

            scores = scores.masked_fill(~mask, float("-inf"))

        attn = F.softmax(scores, dim=-1)

        attn = self.attn_dropout(attn)

        out = torch.matmul(attn, v)

        out = out.transpose(1, 2).contiguous().view(B, Tq, C)

        out = self.out_proj(out)

        out = self.resid_dropout(out)

        return out

Overwriting attention_ed.py


In [28]:
%%writefile hybrid_layers.py

import torch
import torch.nn as nn
import torch.nn.functional as F

from attention_ed import RMSNorm


class FeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):

        return self.net(x)


class FastRecurrentBlock(nn.Module):
    """
    Fast recurrent block.

    Uses PyTorch's cuDNN-accelerated GRU instead of a
    Python loop / custom cumulative-product scan.
    """

    def __init__(self, d_model, d_ff, dropout):

        super().__init__()

        self.ln1 = RMSNorm(d_model)

        self.rnn = nn.GRU(
            input_size=d_model,
            hidden_size=d_model,
            num_layers=1,
            batch_first=True
        )

        self.ln2 = RMSNorm(d_model)

        self.ffn = FeedForward(
            d_model,
            d_ff,
            dropout
        )

    def forward(self, x):

        # pre-norm
        r = self.ln1(x)

        # cuDNN GRU:
        # processes the whole sequence efficiently on GPU
        r, _ = self.rnn(r)

        x = x + r

        x = x + self.ffn(self.ln2(x))

        return x


class FastAttentionBlock(nn.Module):
    """
    Transformer block using fused scaled_dot_product_attention.
    """

    def __init__(
        self,
        d_model,
        n_heads,
        max_seq_len,
        d_ff,
        dropout,
        rope=True,
        qk_norm=True
    ):

        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.ln1 = RMSNorm(d_model)

        self.q_proj = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        self.k_proj = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        self.v_proj = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        self.qk_norm = qk_norm

        if qk_norm:

            self.q_norm = RMSNorm(
                self.d_head
            )

            self.k_norm = RMSNorm(
                self.d_head
            )

        self.rope = rope

        if rope:

            from attention_ed import build_rope_cache

            cos, sin = build_rope_cache(
                max_seq_len,
                self.d_head
            )

            self.register_buffer(
                "rope_cos",
                cos,
                persistent=False
            )

            self.register_buffer(
                "rope_sin",
                sin,
                persistent=False
            )

        self.ln2 = RMSNorm(d_model)

        self.ffn = FeedForward(
            d_model,
            d_ff,
            dropout
        )

        self.dropout = dropout

    def apply_rope(self, x):

        # x: (B, H, T, D)

        cos = self.rope_cos[:x.shape[2]]
        sin = self.rope_sin[:x.shape[2]]

        # reuse implementation from attention_ed
        from attention_ed import apply_rope

        return apply_rope(
            x,
            cos,
            sin
        )

    def forward(self, x):

        B, T, C = x.shape

        r = self.ln1(x)

        q = self.q_proj(r)
        k = self.k_proj(r)
        v = self.v_proj(r)

        q = q.view(
            B,
            T,
            self.n_heads,
            self.d_head
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_heads,
            self.d_head
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_heads,
            self.d_head
        ).transpose(1, 2)

        if self.qk_norm:

            q = self.q_norm(q)
            k = self.k_norm(k)

        if self.rope:

            q = self.apply_rope(q)
            k = self.apply_rope(k)

        # FUSED attention
        r = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True
        )

        r = r.transpose(
            1,
            2
        ).contiguous().view(
            B,
            T,
            C
        )

        r = self.out_proj(r)

        x = x + r

        x = x + self.ffn(
            self.ln2(x)
        )

        return x

Overwriting hybrid_layers.py


In [30]:
%%writefile model_arch_hybrid.py

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

from attention_ed import RMSNorm
from hybrid_layers import (
    FastRecurrentBlock,
    FastAttentionBlock
)


@dataclass
class HybridConfig:

    vocab_size: int

    d_model: int = 768

    n_heads: int = 12

    n_recurrent_layers: int = 8
    n_attention_layers: int = 4

    # chosen to keep model near 105M parameters
    d_ff: int = 2304

    max_seq_len: int = 1024

    dropout: float = 0.0

    rope: bool = True
    qk_norm: bool = True


class HybridLM(nn.Module):

    def __init__(self, cfg):

        super().__init__()

        self.cfg = cfg

        self.tok_emb = nn.Embedding(
            cfg.vocab_size,
            cfg.d_model
        )

        blocks = []

        # [R R A] x 4
        for _ in range(4):

            blocks.append(
                FastRecurrentBlock(
                    cfg.d_model,
                    cfg.d_ff,
                    cfg.dropout
                )
            )

            blocks.append(
                FastRecurrentBlock(
                    cfg.d_model,
                    cfg.d_ff,
                    cfg.dropout
                )
            )

            blocks.append(
                FastAttentionBlock(
                    cfg.d_model,
                    cfg.n_heads,
                    cfg.max_seq_len,
                    cfg.d_ff,
                    cfg.dropout,
                    rope=cfg.rope,
                    qk_norm=cfg.qk_norm
                )
            )

        self.blocks = nn.ModuleList(blocks)

        self.ln_f = RMSNorm(
            cfg.d_model
        )

        self.lm_head = nn.Linear(
            cfg.d_model,
            cfg.vocab_size,
            bias=False
        )

        # weight tying
        self.lm_head.weight = self.tok_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:

                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(self, x):

        x = self.tok_emb(x)

        for block in self.blocks:

            x = block(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        return logits

Overwriting model_arch_hybrid.py


In [32]:
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader, Subset


TRAIN_PATH = "/kaggle/input/models/strugglingug/model6/pytorch/default/1/train_v3.bin"
VAL_PATH = "/kaggle/input/models/strugglingug/model6/pytorch/default/1/val_v3.bin"

BLOCK_SIZE = 1024

BATCH_SIZE = 4

ACCUM_STEPS = 2


class HybridDataset(Dataset):

    def __init__(
        self,
        path,
        block_size
    ):

        self.tokens = np.memmap(
            path,
            dtype=np.uint16,
            mode="r"
        )

        self.block_size = block_size

    def __len__(self):

        return len(self.tokens) - self.block_size

    def __getitem__(self, idx):

        arr = np.asarray(
            self.tokens[
                idx:idx + self.block_size + 1
            ],
            dtype=np.int64
        )

        x = torch.from_numpy(
            arr[:-1]
        )

        y = torch.from_numpy(
            arr[1:]
        )

        return x, y


train_dataset = HybridDataset(
    TRAIN_PATH,
    BLOCK_SIZE
)

val_dataset = HybridDataset(
    VAL_PATH,
    BLOCK_SIZE
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)


val_idx = np.random.default_rng(0).integers(
    0,
    len(val_dataset),
    size=40
).tolist()


val_loader = DataLoader(
    Subset(
        val_dataset,
        val_idx
    ),
    batch_size=BATCH_SIZE,
    shuffle=False
)


print(
    "Train windows:",
    len(train_dataset)
)

print(
    "Validation windows:",
    len(val_dataset)
)

Train windows: 145028515
Validation windows: 4969437


In [33]:
import importlib

import hybrid_layers
import model_arch_hybrid

importlib.reload(hybrid_layers)
importlib.reload(model_arch_hybrid)

from model_arch_hybrid import (
    HybridLM,
    HybridConfig
)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


cfg = HybridConfig(
    vocab_size=tokenizer.vocab_size,
    d_model=768,
    n_heads=12,
    n_recurrent_layers=8,
    n_attention_layers=4,
    d_ff=2304,
    max_seq_len=1024,
    dropout=0.0,
    rope=True,
    qk_norm=True
)

model = HybridLM(cfg).to(device)

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters()) / 1e6,
    "M"
)

print(
    "Device:",
    device
)

Parameters: 103.349504 M
Device: cuda


In [31]:
import importlib
import hybrid_layers
import model_arch_hybrid

importlib.reload(hybrid_layers)
importlib.reload(model_arch_hybrid)

from model_arch_hybrid import HybridLM, HybridConfig

In [21]:
model = HybridLM(cfg).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=6e-4,
    weight_decay=0.1,
    betas=(0.9, 0.95)
)

In [26]:
model = HybridLM(cfg).to(device)

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

print("Device:", device)

Parameters: 103359488
Device: cuda


In [11]:
import time
import math

import torch
import torch.nn.functional as F

from torch.utils.data import DataLoader


V = tokenizer.vocab_size

WEIGHT_DECAY = 0.1

BETAS = (0.9, 0.95)

GRAD_CLIP = 1.0

USE_AMP = True


RUN_SWEEP = True

SWEEP_LRS = [
    3e-4,
    6e-4,
    1e-3,
    2e-3
]

SWEEP_STEPS = 200

LR_SAFETY = 0.8


def make_lr_fn(
    peak_lr,
    total_steps,
    warmup_steps,
    min_ratio=0.1
):

    def fn(step):

        if step <= warmup_steps:

            return (
                peak_lr
                * step
                / warmup_steps
            )

        p = (
            step - warmup_steps
        ) / max(
            1,
            total_steps - warmup_steps
        )

        return peak_lr * (
            min_ratio
            +
            (1 - min_ratio)
            *
            0.5
            *
            (
                1
                +
                math.cos(
                    math.pi * p
                )
            )
        )

    return fn


@torch.no_grad()
def evaluate(m):

    m.eval()

    losses = []

    for xv, yv in val_loader:

        xv = xv.to(device)
        yv = yv.to(device)

        with torch.autocast(
            "cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            logits = m(xv)

        loss = F.cross_entropy(
            logits.float().reshape(-1, V),
            yv.reshape(-1)
        )

        losses.append(
            loss.item()
        )

        del logits

    m.train()

    return sum(losses) / len(losses)


def sweep_trial(
    lr,
    steps
):

    torch.manual_seed(0)

    m = HybridLM(
        cfg
    ).to(device)

    m.train()

    opt = torch.optim.AdamW(
        m.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY,
        betas=BETAS
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    lr_fn = make_lr_fn(
        lr,
        steps,
        max(
            20,
            int(0.05 * steps)
        )
    )

    loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(0)
    )

    it = iter(loader)

    val = float("nan")

    for step in range(
        1,
        steps + 1
    ):

        for g in opt.param_groups:

            g["lr"] = lr_fn(step)

        opt.zero_grad(
            set_to_none=True
        )

        total = 0.0

        for _ in range(
            ACCUM_STEPS
        ):

            try:

                x, y = next(it)

            except StopIteration:

                it = iter(loader)

                x, y = next(it)

            x = x.to(device)
            y = y.to(device)

            with torch.autocast(
                "cuda",
                dtype=torch.float16,
                enabled=USE_AMP
            ):

                logits = m(x)

            loss = (
                F.cross_entropy(
                    logits.float().reshape(
                        -1,
                        V
                    ),
                    y.reshape(-1)
                )
                / ACCUM_STEPS
            )

            scaler.scale(
                loss
            ).backward()

            total += loss.item()

            del logits
            del loss

        if not math.isfinite(total):

            print(
                f"lr {lr:g}: "
                f"loss became NaN/inf "
                f"at step {step}"
            )

            break

        scaler.unscale_(opt)

        torch.nn.utils.clip_grad_norm_(
            m.parameters(),
            GRAD_CLIP
        )

        scaler.step(opt)

        scaler.update()

    else:

        val = evaluate(m)

    del m
    del opt

    torch.cuda.empty_cache()

    return val


if RUN_SWEEP:

    results = {}

    for lr in SWEEP_LRS:

        t = time.time()

        results[lr] = sweep_trial(
            lr,
            SWEEP_STEPS
        )

        print(
            f"lr {lr:g} -> "
            f"val loss {results[lr]:.4f} "
            f"({(time.time()-t)/60:.1f} min)"
        )

    valid = {
        k: v
        for k, v in results.items()
        if math.isfinite(v)
    }

    best_lr = min(
        valid,
        key=valid.get
    )

    LEARNING_RATE = (
        best_lr
        * LR_SAFETY
    )

    print(
        f"\nbest sweep LR = "
        f"{best_lr:g}"
    )

    print(
        f"using "
        f"LEARNING_RATE = "
        f"{LEARNING_RATE:.2e}"
    )

else:

    LEARNING_RATE = 6e-4

    print(
        "sweep skipped, "
        "LEARNING_RATE =",
        LEARNING_RATE
    )

lr 0.0003 -> val loss 10.3480 (2.9 min)
lr 0.0006 -> val loss 10.3480 (2.8 min)
lr 0.001 -> val loss 10.3480 (2.8 min)
lr 0.002 -> val loss 10.3480 (2.8 min)

best sweep LR = 0.0003
using LEARNING_RATE = 2.40e-04


In [34]:
import time
import os
import math

import torch
import torch.nn.functional as F


TARGET_TOKENS = 100_000_000


TRAIN_STEPS = (
    TARGET_TOKENS
    //
    (
        BATCH_SIZE
        * ACCUM_STEPS
        * BLOCK_SIZE
    )
)


WARMUP_STEPS = 300

MIN_LR_RATIO = 0.1

CKPT_DIR = (
    "/kaggle/working/"
    "checkpoints_hybrid"
)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=BETAS
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)


train_iter = iter(
    train_loader
)


def lr_at(step):

    if step <= WARMUP_STEPS:

        return (
            LEARNING_RATE
            * step
            / WARMUP_STEPS
        )

    p = (
        step - WARMUP_STEPS
    ) / max(
        1,
        TRAIN_STEPS - WARMUP_STEPS
    )

    return LEARNING_RATE * (
        MIN_LR_RATIO
        +
        (1 - MIN_LR_RATIO)
        *
        0.5
        *
        (
            1
            +
            math.cos(
                math.pi * p
            )
        )
    )


def next_batch(
    it,
    loader
):

    try:

        return next(it), it

    except StopIteration:

        it = iter(loader)

        return next(it), it


def save_ckpt(
    path,
    step
):

    os.makedirs(
        os.path.dirname(path),
        exist_ok=True
    )

    torch.save(
        {
            "model_state":
                model.state_dict(),

            "config":
                vars(cfg),

            "vocab_size":
                V,

            "step":
                step,

            "optimizer_state":
                optimizer.state_dict(),

            "scaler_state":
                scaler.state_dict()
        },
        path
    )


RESUME_FROM = None

start_step = 0


if RESUME_FROM:

    ckpt = torch.load(
        RESUME_FROM,
        map_location=device
    )

    model.load_state_dict(
        ckpt["model_state"]
    )

    start_step = ckpt["step"]

    if "optimizer_state" in ckpt:

        optimizer.load_state_dict(
            ckpt["optimizer_state"]
        )

        scaler.load_state_dict(
            ckpt["scaler_state"]
        )

    print(
        "resumed from step",
        start_step
    )


print(
    "tokens consumed/step:",
    BATCH_SIZE
    * ACCUM_STEPS
    * BLOCK_SIZE
)

print(
    "steps:",
    TRAIN_STEPS
)

print(
    "peak LR:",
    f"{LEARNING_RATE:.2e}"
)

torch.backends.cudnn.benchmark = True

model.train()

t0 = time.time()


for step in range(
    start_step + 1,
    TRAIN_STEPS + 1
):

    for g in optimizer.param_groups:

        g["lr"] = lr_at(step)

    optimizer.zero_grad(
        set_to_none=True
    )

    train_loss = 0.0

    for _ in range(
        ACCUM_STEPS
    ):

        (x, y), train_iter = next_batch(
            train_iter,
            train_loader
        )

        x = x.to(device)
        y = y.to(device)

        with torch.autocast(
            "cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            logits = model(x)

        loss = (
            F.cross_entropy(
                logits.float().reshape(
                    -1,
                    V
                ),
                y.reshape(-1)
            )
            / ACCUM_STEPS
        )

        scaler.scale(
            loss
        ).backward()

        train_loss += loss.item()

        del logits
        del loss

    scaler.unscale_(
        optimizer
    )

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        GRAD_CLIP
    )

    scaler.step(
        optimizer
    )

    scaler.update()

    if step == start_step + 1:

        print(
            f"first loss: "
            f"{train_loss:.3f} "
            f"(expect about 10.1-10.6) | "
            f"peak GPU memory: "
            f"{torch.cuda.max_memory_allocated()/1e9:.1f} GB"
        )

    if step % 10 == 0:

        val_loss = evaluate(
            model
        )

        print(
            f"Step "
            f"{step}/{TRAIN_STEPS} | "
            f"Train "
            f"{train_loss:.4f} | "
            f"Val "
            f"{val_loss:.4f} | "
            f"LR "
            f"{lr_at(step):.2e} | "
            f"{(time.time()-t0)/60:.1f} min"
        )

    if step % 2000 == 0:

        save_ckpt(
            f"{CKPT_DIR}/last.pt",
            step
        )


save_ckpt(
    f"{CKPT_DIR}/hybrid_final.pt",
    TRAIN_STEPS
)

tokens consumed/step: 8192
steps: 12207
peak LR: 2.40e-04
first loss: 10.514 (expect about 10.1-10.6) | peak GPU memory: 7.4 GB
Step 10/12207 | Train 10.1210 | Val 9.9456 | LR 8.00e-06 | 0.5 min
Step 20/12207 | Train 9.3005 | Val 9.1867 | LR 1.60e-05 | 0.8 min
Step 30/12207 | Train 8.8065 | Val 8.7555 | LR 2.40e-05 | 1.0 min
Step 40/12207 | Train 8.2717 | Val 8.4010 | LR 3.20e-05 | 1.2 min
Step 50/12207 | Train 8.0942 | Val 8.0724 | LR 4.00e-05 | 1.5 min
Step 60/12207 | Train 7.8895 | Val 7.7194 | LR 4.80e-05 | 1.7 min
Step 70/12207 | Train 7.5909 | Val 7.3856 | LR 5.60e-05 | 1.9 min
Step 80/12207 | Train 6.9800 | Val 7.0790 | LR 6.40e-05 | 2.2 min
Step 90/12207 | Train 6.6850 | Val 6.8090 | LR 7.20e-05 | 2.4 min
Step 100/12207 | Train 6.3278 | Val 6.5644 | LR 8.00e-05 | 2.6 min
Step 110/12207 | Train 6.2575 | Val 6.3421 | LR 8.80e-05 | 2.9 min
Step 120/12207 | Train 6.3333 | Val 6.1980 | LR 9.60e-05 | 3.1 min
Step 130/12207 | Train 6.0125 | Val 6.0652 | LR 1.04e-04 | 3.3 min
Step 140/

In [35]:
import torch
import torch.nn.functional as F


BOS_ID = tokenizer.eos_id


@torch.no_grad()
def generate_batch_hybrid(
    model,
    src,
    bos_id,
    eos_id,
    temperature=0.8,
    top_k=50,
    rep_penalty=1.1,
    max_new_tokens=400
):

    model.eval()

    B = src.shape[0]

    max_new_tokens = min(
        max_new_tokens,
        model.cfg.max_seq_len - src.shape[1]
    )

    y = src.clone()

    for i in range(
        max_new_tokens
    ):

        # Keep the most recent context
        # if we approach the model limit.
        x = y[
            :,
            -model.cfg.max_seq_len:
        ]

        with torch.autocast(
            "cuda",
            dtype=torch.float16
        ):

            logits = model(
                x
            )[:, -1, :].float()

        # Never generate EOS because the
        # submission requires exactly 128 words.
        logits[:, eos_id] = -float(
            "inf"
        )

        # Repetition penalty
        window = y[:, -256:]

        s = logits.gather(
            1,
            window
        )

        s = torch.where(
            s > 0,
            s / rep_penalty,
            s * rep_penalty
        )

        logits.scatter_(
            1,
            window,
            s
        )

        logits = (
            logits
            / temperature
        )

        # Top-k
        kth = torch.topk(
            logits,
            top_k
        )[0][:, -1:]

        logits[
            logits < kth
        ] = -float("inf")

        probs = F.softmax(
            logits,
            dim=-1
        )

        nxt = torch.multinomial(
            probs,
            1
        )

        y = torch.cat(
            [y, nxt],
            dim=1
        )

        # Stop once every sample has
        # enough words.
        if (
            (i + 1) % 16 == 0
            and all(
                len(
                    tokenizer.decode(
                        r[src.shape[1]:].tolist()
                    ).split()
                ) > 128
                for r in y
            )
        ):

            break

    return [
        tokenizer.decode(
            r[src.shape[1]:].tolist()
        )
        for r in y
    ]

In [38]:
from tqdm.auto import tqdm


SMOKE = False

BATCH = 16


torch.manual_seed(42)


keys = (
    list(test_data)[:16]
    if SMOKE
    else list(test_data)
)


ids = {
    k: tokenizer.encode(
        test_data[k]["text"]
    )
    for k in keys
}


# Keep at most 512 prompt tokens so
# there is room for the continuation.
L = min(
    BLOCK_SIZE // 2,
    min(
        len(v)
        for v in ids.values()
    )
)


print(
    "source length used:",
    L,
    "tokens"
)


predictions = {}


for i in tqdm(
    range(
        0,
        len(keys),
        BATCH
    )
):

    kb = keys[
        i:i+BATCH
    ]

    src = torch.tensor(
        [
            ids[k][-L:]
            for k in kb
        ],
        device=device
    )

    generated = generate_batch_hybrid(
        model,
        src,
        BOS_ID,
        tokenizer.eos_id
    )

    for k, t in zip(
        kb,
        generated
    ):

        predictions[k] = {
            "text":
                " ".join(
                    t.split()[:128]
                )
        }


if SMOKE:

    for k in keys[:2]:

        print(
            predictions[k]["text"],
            "\n---"
        )

source length used: 512 tokens


  0%|          | 0/63 [00:00<?, ?it/s]

In [37]:
import json

from huggingface_hub import (
    hf_hub_download,
    HfApi
)


REPO = "Exploration-Lab/CS-781-Capstone"


files = [
    f
    for f in HfApi().list_repo_files(
        REPO,
        repo_type="dataset"
    )
    if "Phase_1" in f
    and "Test" in f
]


print("Test files found:")
print(files)


TEST_FILE = files[0]


test_path = hf_hub_download(
    REPO,
    TEST_FILE,
    repo_type="dataset"
)


with open(
    test_path,
    "r",
    encoding="utf-8"
) as f:

    test_data = json.load(f)


print(
    "Loaded",
    len(test_data),
    "test prompts"
)

Test files found:
['Phase_1_Pretraining/Test/test.json']


test.json: 0.00B [00:00, ?B/s]

Loaded 1000 test prompts


In [39]:
import zipfile


# ------------------------------------------------
# Repair predictions shorter than 128 words
# ------------------------------------------------

for attempt in range(5):

    bad = [
        k
        for k, v in predictions.items()
        if len(
            v["text"].split()
        ) != 128
    ]

    if not bad:

        break

    print(
        f"attempt {attempt + 1}: "
        f"regenerating "
        f"{len(bad)} prediction(s)"
    )

    torch.manual_seed(
        1000 + attempt
    )

    for i in range(
        0,
        len(bad),
        BATCH
    ):

        kb = bad[
            i:i+BATCH
        ]

        src = torch.tensor(
            [
                ids[k][-L:]
                for k in kb
            ],
            device=device
        )

        generated = generate_batch_hybrid(
            model,
            src,
            BOS_ID,
            tokenizer.eos_id,
            max_new_tokens=500
        )

        for k, t in zip(
            kb,
            generated
        ):

            words = t.split()

            if len(words) >= 128:

                predictions[k] = {
                    "text":
                        " ".join(
                            words[:128]
                        )
                }

            elif len(words) > len(
                predictions[k]["text"].split()
            ):

                predictions[k] = {
                    "text":
                        " ".join(words)
                }


# ------------------------------------------------
# Last-resort repair
# ------------------------------------------------

for k in [
    k
    for k, v in predictions.items()
    if len(v["text"].split()) != 128
]:

    words = predictions[k]["text"].split()

    if words:

        predictions[k] = {
            "text":
                " ".join(
                    (
                        words
                        *
                        (
                            128
                            // len(words)
                            + 1
                        )
                    )[:128]
                )
        }


# ------------------------------------------------
# Validate
# ------------------------------------------------

TAG = "hybrid_minGRU_RRA_T0.8_k50"


assert not SMOKE, (
    "set SMOKE = False "
    "and re-run the generation cell first"
)


assert set(predictions) == set(
    test_data
)


bad = [
    k
    for k, v in predictions.items()
    if len(
        v["text"].split()
    ) != 128
]


assert not bad, (
    f"{len(bad)} predictions "
    f"are not exactly 128 words, "
    f"e.g. {bad[:3]}"
)


# ------------------------------------------------
# Write JSON
# ------------------------------------------------

with open(
    "prediction.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        predictions,
        f,
        ensure_ascii=False
    )


# ------------------------------------------------
# Zip
# ------------------------------------------------

zip_name = (
    f"submission_{TAG}.zip"
)


with zipfile.ZipFile(
    zip_name,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    z.write(
        "prediction.json",
        arcname="prediction.json"
    )


print(
    zipfile.ZipFile(
        zip_name
    ).namelist()
)

attempt 1: regenerating 6 prediction(s)
attempt 2: regenerating 1 prediction(s)
['prediction.json']


In [40]:
from IPython.display import FileLink, display

display(FileLink(
    "checkpoints_hybrid/hybrid_final.pt",
    result_html_prefix="Download checkpoint: "
))

/kaggle/working/checkpoints_hybrid/hybrid_final.pt

In [30]:
%%writefile lightweight_layers.py

import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):

    def __init__(
        self,
        dim,
        eps=1e-6
    ):

        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )

        self.eps = eps

    def forward(self, x):

        rms = x.float().pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return x * self.weight.to(x.dtype)


def get_alibi_slopes(n_heads):

    def power_of_2_slopes(n):

        start = 2 ** (
            -2 ** -(math.log2(n) - 3)
        )

        ratio = start

        return torch.tensor(
            [
                start * ratio ** i
                for i in range(n)
            ],
            dtype=torch.float32
        )

    if math.log2(n_heads).is_integer():

        return power_of_2_slopes(n_heads)

    closest_power = 2 ** int(
        math.floor(math.log2(n_heads))
    )

    extra = n_heads - closest_power

    slopes = power_of_2_slopes(
        closest_power
    )

    extra_slopes = get_alibi_slopes(
        2 * closest_power
    )[0::2]

    return torch.cat(
        [
            slopes,
            extra_slopes[:extra]
        ]
    )


class SwiGLU(nn.Module):

    def __init__(
        self,
        d_model,
        d_ff,
        dropout
    ):

        super().__init__()

        self.w1 = nn.Linear(
            d_model,
            d_ff,
            bias=False
        )

        self.w2 = nn.Linear(
            d_model,
            d_ff,
            bias=False
        )

        self.w3 = nn.Linear(
            d_ff,
            d_model,
            bias=False
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):

        return self.dropout(
            self.w3(
                F.silu(
                    self.w1(x)
                )
                * self.w2(x)
            )
        )


class LocalGQAAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_q_heads,
        n_kv_heads,
        max_seq_len,
        window_size,
        dropout,
        qk_norm=True
    ):

        super().__init__()

        assert d_model % n_q_heads == 0
        assert n_q_heads % n_kv_heads == 0

        self.d_model = d_model

        self.n_q_heads = n_q_heads
        self.n_kv_heads = n_kv_heads

        self.groups = (
            n_q_heads // n_kv_heads
        )

        self.d_head = (
            d_model // n_q_heads
        )

        self.window_size = min(
            window_size,
            max_seq_len
        )

        self.q_proj = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.d_head,
            bias=False
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.d_head,
            bias=False
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        self.qk_norm = qk_norm

        if qk_norm:

            self.q_norm = RMSNorm(
                self.d_head
            )

            self.k_norm = RMSNorm(
                self.d_head
            )

        self.dropout = dropout

        # -------------------------
        # ALiBi
        # -------------------------

        slopes = get_alibi_slopes(
            n_q_heads
        )

        self.register_buffer(
            "alibi_slopes",
            slopes,
            persistent=False
        )

        # distance inside the local window:
        #
        # [255, 254, ..., 1, 0]
        #
        distance = torch.arange(
            self.window_size - 1,
            -1,
            -1,
            dtype=torch.float32
        )

        self.register_buffer(
            "distance",
            distance,
            persistent=False
        )

        # valid positions for each query
        #
        # early tokens have fewer than
        # window_size valid keys.
        query_pos = torch.arange(
            max_seq_len
        ).unsqueeze(1)

        window_pos = torch.arange(
            self.window_size
        ).unsqueeze(0)

        min_valid = (
            self.window_size
            - 1
            - query_pos
        )

        valid = (
            window_pos >= min_valid
        )

        self.register_buffer(
            "valid_mask",
            valid,
            persistent=False
        )

    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_q_heads,
            self.d_head
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_heads,
            self.d_head
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_heads,
            self.d_head
        ).transpose(1, 2)

        if self.qk_norm:

            q = self.q_norm(q)
            k = self.k_norm(k)

        # -------------------------------------------------
        # Group query heads
        #
        # q:
        #   (B, H_kv, groups, T, D)
        #
        # k/v:
        #   (B, H_kv, T, D)
        # -------------------------------------------------

        q = q.view(
            B,
            self.n_kv_heads,
            self.groups,
            T,
            self.d_head
        )

        # -------------------------------------------------
        # Build sliding windows for K and V
        # -------------------------------------------------

        W = self.window_size

        k_pad = F.pad(
            k,
            (0, 0, W - 1, 0)
        )

        v_pad = F.pad(
            v,
            (0, 0, W - 1, 0)
        )

        k_win = k_pad.unfold(
            2,
            W,
            1
        )

        v_win = v_pad.unfold(
            2,
            W,
            1
        )

        # unfold produces:
        #
        # (B, Hkv, T, D, W)
        #
        # convert to:
        #
        # (B, Hkv, T, W, D)

        k_win = k_win.permute(
            0, 1, 2, 4, 3
        )

        v_win = v_win.permute(
            0, 1, 2, 4, 3
        )

        # -------------------------------------------------
        # Local attention scores
        # -------------------------------------------------

        scores = torch.einsum(
            "bhgtd,bhtwd->bhgtw",
            q,
            k_win
        )

        scores = scores / math.sqrt(
            self.d_head
        )

        # -------------------------------------------------
        # ALiBi
        #
        # bias = - slope * distance
        # -------------------------------------------------

        slopes = self.alibi_slopes.view(
            1,
            self.n_kv_heads,
            self.groups,
            1,
            1
        )

        distance = self.distance.view(
            1,
            1,
            1,
            1,
            W
        )

        scores = scores - (
            slopes * distance
        )

        # -------------------------------------------------
        # Mask padding / future-invalid positions
        # -------------------------------------------------

        mask = self.valid_mask[
            :T,
            :W
        ].view(
            1,
            1,
            1,
            T,
            W
        )

        scores = scores.masked_fill(
            ~mask,
            float("-inf")
        )

        # softmax in FP32 for stability
        attn = F.softmax(
            scores.float(),
            dim=-1
        ).to(scores.dtype)

        if self.training and self.dropout > 0:

            attn = F.dropout(
                attn,
                p=self.dropout
            )

        # -------------------------------------------------
        # Weighted values
        # -------------------------------------------------

        out = torch.einsum(
            "bhgtw,bhtwd->bhgtd",
            attn,
            v_win
        )

        out = out.reshape(
            B,
            self.n_q_heads,
            T,
            self.d_head
        )

        out = out.transpose(
            1,
            2
        ).contiguous()

        out = out.view(
            B,
            T,
            C
        )

        return self.out_proj(out)


class LightweightBlock(nn.Module):

    def __init__(
        self,
        d_model,
        n_q_heads,
        n_kv_heads,
        max_seq_len,
        window_size,
        d_ff,
        dropout,
        qk_norm
    ):

        super().__init__()

        self.ln1 = RMSNorm(
            d_model
        )

        self.attn = LocalGQAAttention(
            d_model,
            n_q_heads,
            n_kv_heads,
            max_seq_len,
            window_size,
            dropout,
            qk_norm
        )

        self.ln2 = RMSNorm(
            d_model
        )

        self.ffn = SwiGLU(
            d_model,
            d_ff,
            dropout
        )

    def forward(self, x):

        x = x + self.attn(
            self.ln1(x)
        )

        x = x + self.ffn(
            self.ln2(x)
        )

        return x

Overwriting lightweight_layers.py


In [31]:
%%writefile model_arch_lightweight.py

from dataclasses import dataclass

import torch
import torch.nn as nn

from lightweight_layers import (
    RMSNorm,
    LightweightBlock
)


@dataclass
class LightweightConfig:

    vocab_size: int

    d_model: int = 1024

    n_q_heads: int = 16
    n_kv_heads: int = 4

    d_ff: int = 4096

    max_seq_len: int = 1024

    window_size: int = 128

    dropout: float = 0.0

    qk_norm: bool = True

    # 4 unique blocks
    n_unique_blocks: int = 4

    # each block is executed twice
    n_repeats: int = 2


class LightweightLM(nn.Module):

    def __init__(self, cfg):

        super().__init__()

        self.cfg = cfg

        # -------------------------------------------------
        # Token embedding
        # -------------------------------------------------

        self.tok_emb = nn.Embedding(
            cfg.vocab_size,
            cfg.d_model
        )

        # -------------------------------------------------
        # 4 unique blocks
        #
        # Each block is reused twice.
        #
        # Logical architecture:
        #
        # B1 B2 B3 B4 B1 B2 B3 B4
        # -------------------------------------------------

        self.blocks = nn.ModuleList(
            [
                LightweightBlock(
                    cfg.d_model,
                    cfg.n_q_heads,
                    cfg.n_kv_heads,
                    cfg.max_seq_len,
                    cfg.window_size,
                    cfg.d_ff,
                    cfg.dropout,
                    cfg.qk_norm
                )
                for _ in range(
                    cfg.n_unique_blocks
                )
            ]
        )

        self.ln_f = RMSNorm(
            cfg.d_model
        )

        self.lm_head = nn.Linear(
            cfg.d_model,
            cfg.vocab_size,
            bias=False
        )

        # weight tying
        self.lm_head.weight = (
            self.tok_emb.weight
        )

        self.apply(
            self._init_weights
        )

    def _init_weights(self, module):

        if isinstance(
            module,
            nn.Linear
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:

                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(
            module,
            nn.Embedding
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(self, x):

        x = self.tok_emb(x)

        for _ in range(
            self.cfg.n_repeats
        ):

            for block in self.blocks:

                x = block(x)

        x = self.ln_f(x)

        return self.lm_head(x)

Overwriting model_arch_lightweight.py


In [32]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [33]:
import numpy as np
import torch

from torch.utils.data import (
    Dataset,
    DataLoader,
    Subset
)


TRAIN_PATH = (
    "/kaggle/input/models/"
    "strugglingug/model6/pytorch/"
    "default/1/train_v3.bin"
)

VAL_PATH = (
    "/kaggle/input/models/"
    "strugglingug/model6/pytorch/"
    "default/1/val_v3.bin"
)

BLOCK_SIZE = 1024

BATCH_SIZE = 2


class CausalDataset(Dataset):

    def __init__(
        self,
        path,
        block_size
    ):

        self.tokens = np.memmap(
            path,
            dtype=np.uint16,
            mode="r"
        )

        self.block_size = block_size

    def __len__(self):

        return (
            len(self.tokens)
            - self.block_size
            - 1
        )

    def __getitem__(self, idx):

        arr = np.asarray(
            self.tokens[
                idx:
                idx + self.block_size + 1
            ],
            dtype=np.int64
        )

        x = torch.from_numpy(
            arr[:-1]
        )

        y = torch.from_numpy(
            arr[1:]
        )

        return x, y


train_dataset = CausalDataset(
    TRAIN_PATH,
    BLOCK_SIZE
)

val_dataset = CausalDataset(
    VAL_PATH,
    BLOCK_SIZE
)


# fixed validation samples
rng = np.random.default_rng(0)

val_idx = rng.integers(
    0,
    len(val_dataset),
    size=40
).tolist()


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)


val_loader = DataLoader(
    Subset(
        val_dataset,
        val_idx
    ),
    batch_size=BATCH_SIZE,
    shuffle=False
)


'''
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=1,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    Subset(
        val_dataset,
        val_idx
    ),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=1,
    pin_memory=True,
    persistent_workers=True
)

'''

print(
    "Train windows:",
    len(train_dataset)
)

print(
    "Validation windows:",
    len(val_dataset)
)

Train windows: 145028514
Validation windows: 4969436


In [34]:
import importlib

import lightweight_layers
import model_arch_lightweight

importlib.reload(
    lightweight_layers
)

importlib.reload(
    model_arch_lightweight
)

from model_arch_lightweight import (
    LightweightLM,
    LightweightConfig
)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


cfg = LightweightConfig(

    vocab_size=tokenizer.vocab_size,

    d_model=1024,

    n_q_heads=16,
    n_kv_heads=4,

    d_ff=4096,

    max_seq_len=BLOCK_SIZE,

    window_size=256,

    dropout=0.0,

    qk_norm=True,

    n_unique_blocks=4,

    n_repeats=2
)


model = LightweightLM(
    cfg
).to(device)


params = sum(
    p.numel()
    for p in model.parameters()
)


print(
    "Parameters:",
    round(params / 1e6, 2),
    "M"
)

print(
    "Device:",
    device
)

Parameters: 91.55 M
Device: cuda


In [35]:
import os
import math
import time

import torch
import torch.nn.functional as F


TARGET_TOKENS = 145_000_000

ACCUM_STEPS = 4

TOKENS_PER_UPDATE = (
    BATCH_SIZE
    * BLOCK_SIZE
    * ACCUM_STEPS
)

TRAIN_STEPS = (
    TARGET_TOKENS
    // TOKENS_PER_UPDATE
)


WARMUP_STEPS = 300

LEARNING_RATE = 6e-4

WEIGHT_DECAY = 0.1

BETAS = (
    0.9,
    0.95
)

GRAD_CLIP = 1.0

USE_AMP = True

CKPT_DIR = (
    "/kaggle/working/"
    "checkpoints_lightweight"
)

os.makedirs(
    CKPT_DIR,
    exist_ok=True
)


print(
    "Tokens/update:",
    TOKENS_PER_UPDATE
)

print(
    "Training steps:",
    TRAIN_STEPS
)

Tokens/update: 8192
Training steps: 17700


In [36]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=BETAS
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)


def lr_at(step):

    if step <= WARMUP_STEPS:

        return (
            LEARNING_RATE
            * step
            / WARMUP_STEPS
        )

    p = (
        step - WARMUP_STEPS
    ) / max(
        1,
        TRAIN_STEPS
        - WARMUP_STEPS
    )

    return LEARNING_RATE * (
        0.1
        + 0.9
        * 0.5
        * (
            1
            + math.cos(
                math.pi * p
            )
        )
    )

In [37]:
@torch.no_grad()
def evaluate(model):

    model.eval()

    losses = []

    for xv, yv in val_loader:

        xv = xv.to(
            device,
            non_blocking=True
        )

        yv = yv.to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            "cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            logits = model(xv)

        loss = F.cross_entropy(
            logits.float().reshape(
                -1,
                tokenizer.vocab_size
            ),
            yv.reshape(-1)
        )

        losses.append(
            loss.item()
        )

        del logits

    model.train()

    return sum(losses) / len(losses)

In [38]:
torch.backends.cudnn.benchmark = True

train_iter = iter(
    train_loader
)

model.train()

t0 = time.time()


for step in range(
    1,
    TRAIN_STEPS + 1
):

    lr = lr_at(step)

    for group in optimizer.param_groups:

        group["lr"] = lr


    optimizer.zero_grad(
        set_to_none=True
    )

    train_loss = 0.0


    for _ in range(
        ACCUM_STEPS
    ):

        try:

            x, y = next(
                train_iter
            )

        except StopIteration:

            train_iter = iter(
                train_loader
            )

            x, y = next(
                train_iter
            )


        x = x.to(
            device,
            non_blocking=True
        )

        y = y.to(
            device,
            non_blocking=True
        )


        with torch.autocast(
            "cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            logits = model(x)

            loss = F.cross_entropy(
                logits.float().reshape(
                    -1,
                    tokenizer.vocab_size
                ),
                y.reshape(-1)
            )

            loss = (
                loss / ACCUM_STEPS
            )


        scaler.scale(
            loss
        ).backward()


        train_loss += loss.item()


        del logits
        del loss


    scaler.unscale_(
        optimizer
    )


    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        GRAD_CLIP
    )


    scaler.step(
        optimizer
    )

    scaler.update()


    if step == 1:

        print(
            f"First loss: "
            f"{train_loss:.4f} | "
            f"LR: {lr:.2e}"
        )


    if step % 100 == 0:

        val_loss = evaluate(
            model
        )

        print(
            f"Step {step}/{TRAIN_STEPS} | "
            f"Train {train_loss:.4f} | "
            f"Val {val_loss:.4f} | "
            f"LR {lr:.2e} | "
            f"Time {(time.time()-t0)/60:.1f} min"
        )


    if step % 2000 == 0:

        torch.save(
            {
                "model_state":
                    model.state_dict(),

                "config":
                    vars(cfg),

                "vocab_size":
                    tokenizer.vocab_size,

                "step":
                    step
            },
            f"{CKPT_DIR}/last.pt"
        )

        print(
            "Checkpoint saved."
        )


torch.save(
    {
        "model_state":
            model.state_dict(),

        "config":
            vars(cfg),

        "vocab_size":
            tokenizer.vocab_size,

        "step":
            TRAIN_STEPS
    },
    f"{CKPT_DIR}/lightweight_final.pt"
)


print(
    "Training finished."
)

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 14.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.33 GiB is allocated by PyTorch, and 91.00 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import torch.nn.functional as F


BOS_ID = tokenizer.eos_id


@torch.no_grad()
def generate_batch_lightweight(
    model,
    src,
    bos_id,
    eos_id,
    temperature=0.8,
    top_k=50,
    rep_penalty=1.1,
    max_new_tokens=400
):

    model.eval()

    B = src.shape[0]

    y = torch.full(
        (B, 1),
        bos_id,
        dtype=torch.long,
        device=device
    )

    context = torch.cat(
        [src, y],
        dim=1
    )


    max_new_tokens = min(
        max_new_tokens,
        model.cfg.max_seq_len
        - src.shape[1]
    )


    for _ in range(
        max_new_tokens
    ):

        x = context[
            :,
            -model.cfg.max_seq_len:
        ]


        with torch.autocast(
            "cuda",
            dtype=torch.float16
        ):

            logits = model(x)


        logits = logits[
            :, -1, :
        ].float()


        # repetition penalty
        if rep_penalty != 1.0:

            for b in range(B):

                seen = torch.unique(
                    y[b]
                )

                vals = logits[
                    b,
                    seen
                ]

                logits[
                    b,
                    seen
                ] = torch.where(
                    vals < 0,
                    vals * rep_penalty,
                    vals / rep_penalty
                )


        # We need exactly 128 words.
        # Don't allow EOS to terminate generation early.
        logits[:, eos_id] = -float(
            "inf"
        )


        logits = (
            logits
            / temperature
        )


        if top_k is not None:

            k = min(
                top_k,
                logits.shape[-1]
            )

            values, indices = torch.topk(
                logits,
                k
            )

            filtered = torch.full_like(
                logits,
                -float("inf")
            )

            filtered.scatter_(
                1,
                indices,
                values
            )

            logits = filtered


        probs = F.softmax(
            logits,
            dim=-1
        )


        next_token = torch.multinomial(
            probs,
            num_samples=1
        )


        y = torch.cat(
            [
                y,
                next_token
            ],
            dim=1
        )


        context = torch.cat(
            [
                src,
                y
            ],
            dim=1
        )


    generated = []

    for b in range(B):

        text = tokenizer.decode(
            y[b].tolist()
        )

        generated.append(
            text
        )

    model.train()

    return generated

In [ ]:
from tqdm.auto import tqdm


SMOKE = True

BATCH = 16


torch.manual_seed(42)


keys = (
    list(test_data)[:16]
    if SMOKE
    else list(test_data)
)


ids = {
    k: tokenizer.encode(
        test_data[k]["text"]
    )
    for k in keys
}


# Use at most 512 prompt tokens.
L = min(
    BLOCK_SIZE // 2,
    min(
        len(v)
        for v in ids.values()
    )
)


print(
    "source length used:",
    L,
    "tokens"
)


predictions = {}


for i in tqdm(
    range(
        0,
        len(keys),
        BATCH
    )
):

    kb = keys[
        i:i+BATCH
    ]


    src = torch.tensor(
        [
            ids[k][-L:]
            for k in kb
        ],
        dtype=torch.long,
        device=device
    )


    generated = (
        generate_batch_lightweight(
            model,
            src,
            BOS_ID,
            tokenizer.eos_id
        )
    )


    for k, t in zip(
        kb,
        generated
    ):

        predictions[k] = {
            "text":
                " ".join(
                    t.split()[:128]
                )
        }


if SMOKE:

    for k in keys[:2]:

        print(
            predictions[k]["text"],
            "\n---"
        )